In [ ]:
# ── Pipeline Mode Selector ────────────────────────────────────────────────────
# Set these flags before running the notebook.

RUN_MODE = "stable"   # "stable" | "experimental" | "redrongo"

# stable:       Custom backbone, standard MCMC, self-training
#               Produces defensible results for AI Village submission
# experimental: DINOv2 backbone, cross-script similarity, sequential embeddings
#               Slower, less stable, for research exploration
# redrongo:     Stable pipeline + RedRongo agent
#               For live demo at AI Village poster

SKIP_ZONE_A_TRAINING = False   # True if embeddings_cache.pt already in Drive
SKIP_ZONE_B = False            # True if LMs and passages already in Drive
SKIP_ZONE_C = False            # True if ranking.json already in Drive

print(f"Run mode: {RUN_MODE}")
print(f"Skip Zone A training: {SKIP_ZONE_A_TRAINING}")

In [ ]:
# ── Setup Cell 1: Mount Google Drive (run once per session) ──────────────────
# Drive is the persistence layer: checkpoints, LMs, and outputs survive
# session restarts here even though /content/ is wiped on disconnect.
#
# Before running: Runtime → Change runtime type → T4 GPU

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
Path(CHECKPOINTS_DIR).mkdir(parents=True, exist_ok=True)
print(f"Drive mounted. Checkpoints directory: {CHECKPOINTS_DIR}")
# ── MLflow experiment tracking (Drive-backed, survives session restarts) ──────
MLFLOW_TRACKING_DIR = "/content/drive/MyDrive/hackingrongo_mlruns"
Path(MLFLOW_TRACKING_DIR).mkdir(parents=True, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = f"file://{MLFLOW_TRACKING_DIR}"
print(f"MLflow tracking URI: {os.environ['MLFLOW_TRACKING_URI']}")


In [ ]:
# ── Setup Cell 2: Clone repo and install package ─────────────────────────────
# Clones to /content/repo (ephemeral). Re-run at the start of every session.
# Before running: Runtime → Change runtime type → T4 GPU
#
# Uses --no-deps to avoid Colab pip resolver conflicts: torch/torchvision/
# numpy/scipy are already installed at GPU-compatible versions by Colab and
# should not be upgraded. Setup Cell 3 installs remaining non-Colab dependencies.

import os, subprocess, sys

REPO_URL = "https://github.com/sperkswerks-ai/hackingrongo.git"
REPO_DIR = "/content/repo"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin"])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"])
    print("Repo updated to latest commit.")

os.chdir(REPO_DIR)
# --no-deps: register the package in editable mode without touching Colab's
# pre-installed torch/torchvision/numpy/scipy (Setup Cell 3 handles everything else).
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"
])
print("Package installed. Working directory:", os.getcwd())

In [ ]:
# ── Setup Cell 3: Extra dependencies ───────────────────────────────────────────
# Installs packages not pre-loaded by Colab's base image.
# torch / torchvision / numpy / scipy / Pillow are already present — skip them.
# Packages are installed in separate calls so a single failure is identifiable.
#
# SA solver uses a pure numpy fallback — no D-Wave packages required.

import subprocess, sys

def pip(*args):
    """Run pip install; print output only on failure."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"FAILED: pip install {' '.join(args)}")
        print(result.stdout[-2000:] if result.stdout else "")
        print(result.stderr[-2000:] if result.stderr else "")
        raise RuntimeError(f"pip install failed for: {args}")
    print(f"  ✓ {', '.join(args)}")

pip("hydra-core>=1.3", "omegaconf>=2.3")
pip("click>=8.1", "jinja2>=3.1")
pip("dimod>=0.12")
pip("cairosvg")
pip("umap-learn", "hdbscan")
pip("gdown")
pip("dwave-samplers>=1.0")  # provides dwave.samplers.SimulatedAnnealingSampler (neal replacement)

print("\nAll extra dependencies installed.")

In [ ]:
# ── Setup Cell 4: Download data and restore checkpoints ───────────────────────
import subprocess, shutil, zipfile, os
from pathlib import Path

REPO_DIR = "/content/repo"
DRIVE_FILE_ID = "1NGg7459yIo1Xky-rvDBwSMqZ0rYTFb_X"
ZIP_PATH = Path("/content/hackingrongo_data.zip")
DATA_DIR = Path(REPO_DIR) / "data"
FORCE_REEXTRACT = False

if not ZIP_PATH.exists():
    print("Downloading zip from Google Drive...")
    subprocess.check_call(["gdown", DRIVE_FILE_ID, "-O", str(ZIP_PATH)])
    print("Download complete.")
else:
    print("Zip already present — skipping download.")

_sentinel = DATA_DIR / "corpus" / "A.json"
if _sentinel.exists() and not FORCE_REEXTRACT:
    n_corpus = len(list((DATA_DIR / "corpus").glob("*.json")))
    print(f"Corpus already extracted ({n_corpus} JSON files) — skipping.")
    print("  Set FORCE_REEXTRACT = True to overwrite.")
else:
    TMPDIR = Path("/content/hackingrongo_extract")
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR.mkdir()
    print(f"Extracting data/ from {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        data_members = [m for m in zf.namelist() if m.startswith("data/")]
        if not data_members:
            raise RuntimeError(
                "Zip contains no data/ folder. Check the zip was created from "
                "inside the hackingrongo/ project root."
            )
        zf.extractall(TMPDIR, members=data_members)
    extracted_data = TMPDIR / "data"
    DATA_DIR.mkdir(exist_ok=True)
    for item in extracted_data.iterdir():
        dest = DATA_DIR / item.name
        if dest.exists():
            shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
        shutil.move(str(item), str(dest))
    shutil.rmtree(TMPDIR)
    n_corpus = len(list((DATA_DIR / "corpus").glob("*.json")))
    svgs  = list(DATA_DIR.rglob("*.svg"))
    crops = list((DATA_DIR / "glyphs" / "3d_crops").rglob("*.png")) if (DATA_DIR / "glyphs" / "3d_crops").exists() else []
    views = list((DATA_DIR / "glyphs" / "synthetic_views").rglob("*.png")) if (DATA_DIR / "glyphs" / "synthetic_views").exists() else []
    print(f"Done: {n_corpus} corpus JSONs, {len(svgs)} SVGs in {DATA_DIR}")
    print(f"  3D crops:        {len(crops)} images (data/glyphs/3d_crops/)")
    print(f"  Synthetic views: {len(views)} images (data/glyphs/synthetic_views/)")

# ── Restore checkpoints from Drive ────────────────────────────────────────────
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
ckpt_path = Path(CHECKPOINTS_DIR)
if ckpt_path.exists():
    restored = 0
    for f in ckpt_path.glob("*.pt"):
        dest = Path(REPO_DIR) / "outputs" / f.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest)
        print(f"  Restored: {f.name}  ({f.stat().st_size/1024/1024:.1f} MB)")
        restored += 1
    lm_dest = DATA_DIR / "language_models"
    lm_dest.mkdir(parents=True, exist_ok=True)
    for f in ckpt_path.glob("*_lm.json"):
        shutil.copy(f, lm_dest / f.name)
        print(f"  Restored LM: {f.name}")
        restored += 1
    pv = ckpt_path / "parallel_variants_auto.json"
    if pv.exists():
        par_dest = DATA_DIR / "parallels"
        par_dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(pv, par_dest / pv.name)
        print("  Restored: parallel_variants_auto.json")
        restored += 1
    print(f"\nRestored {restored} file(s) from Drive.")
else:
    print("No Drive checkpoints found — fresh start.")

print("\nSetup Cell 5 (smoke test) is ready to run.")

In [ ]:
# ── Setup Cell 5: Smoke test — verify imports and data ───────────────────────
# Runs quickly (< 10 seconds). Must pass before running Zone A Cell 1.

import os, json
from pathlib import Path

REPO_DIR = "/content/repo"
DATA_DIR = f"{REPO_DIR}/data"
os.chdir(REPO_DIR)

print("── Import check ──────────────────────────────────────────")
import hackingrongo
from hackingrongo.data.corpus import load_corpus
from hackingrongo.zone_a.autoencoder import ConvAutoencoder
from hackingrongo.zone_b.entropy import index_of_coincidence
from hackingrongo.zone_c.mcmc import MCMCSampler
print("  ✓ All core imports OK")

print("\n── Corpus check ──────────────────────────────────────────")
corpus_dir = Path(f"{DATA_DIR}/corpus")
tablet_files = sorted(corpus_dir.glob("*.json"))
print(f"  Tablet files: {len(tablet_files)}")
assert len(tablet_files) > 0, "No tablet JSON files found — run Setup Cell 4 first"

total_tokens = 0
for tf in tablet_files[:3]:
    d = json.loads(tf.read_text())
    n = len(d.get("glyphs", []))
    total_tokens += n
    print(f"  {tf.stem}: {n} tokens")
print(f"  ✓ Corpus readable")

print("\n── GPU check ─────────────────────────────────────────────")
import torch
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  ⚠ No GPU — Zone A training will be slow. Runtime → Change runtime type → T4 GPU")

print("\n✓ Smoke test passed. Zone A Cell 1 (training) is ready.")

In [ ]:
# ── Zone A Cell 1: Train autoencoder ─────────────────────────────────────────
# Auto-resumes from the latest checkpoint on Drive.
# If 50 epochs are already complete, skips training and re-extracts embeddings.
# To force a full retrain from scratch, set resume_checkpoint to "none" below
# or delete autoencoder_epoch*.pt from the Drive checkpoints directory.
#
# Outputs saved to Drive:
#   autoencoder_epoch*.pt          ← training checkpoints
#   embeddings_cache.pt            ← glyph embedding vectors (input to Zone A Cell 2)

import subprocess, sys, os, shutil
from datetime import datetime, timezone
from pathlib import Path
import torch

STEP_TIMEOUT    = 5400  # seconds; kill and skip if exceeded
REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
EMBEDDINGS_CACHE = Path(f"{REPO_DIR}/outputs/embeddings_cache.pt")
CHECKPOINT       = Path(f"{REPO_DIR}/outputs/checkpoints/zone_a_train.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)

ckpt_dst = Path(REPO_DIR) / "outputs" / "checkpoints"
ckpt_dst.mkdir(parents=True, exist_ok=True)

# ── Restore checkpoints from Drive ────────────────────────────────────────────
# Skip any that don't match the current model architecture.
from hackingrongo.zone_a.autoencoder import ConvAutoencoder
import omegaconf
_cfg = omegaconf.OmegaConf.load(f"{REPO_DIR}/conf/config.yaml")
_model = ConvAutoencoder(_cfg)
_current_keys = set(_model.state_dict().keys())
del _model, _cfg

for f in sorted(Path(CHECKPOINTS_DIR).glob("autoencoder_*.pt")):
    try:
        ckpt = torch.load(f, map_location="cpu", weights_only=True)
        ckpt_keys = set(ckpt["model_state_dict"].keys())
        if ckpt_keys != _current_keys:
            raise RuntimeError("key mismatch: old arch")
        shutil.copy(f, ckpt_dst / f.name)
        print(f"  Restored checkpoint: {f.name}")
    except Exception as e:
        print(f"  Skipped incompatible checkpoint {f.name}: {e}")

# ── Pre-flight: validate or purge existing embeddings cache ───────────────────
print("=" * 60)
print("PRE-FLIGHT: Checking embeddings cache")
print("=" * 60)

CACHE_VALID = False
if EMBEDDINGS_CACHE.exists():
    try:
        _d = torch.load(EMBEDDINGS_CACHE, map_location="cpu", weights_only=True)
        _has_embeddings = isinstance(_d, dict) and "embeddings" in _d
        _has_codes      = _has_embeddings and any(_d.get("barthel_codes", []))
        if _has_embeddings and _has_codes:
            _n       = len(_d["embeddings"])
            _n_coded = sum(1 for c in _d["barthel_codes"] if c)
            print(f"  Cache OK: {_n} embeddings, {_n_coded} with Barthel codes.")
            print("  Skipping training — delete embeddings_cache.pt on Drive to retrain.")
            CACHE_VALID = True
        else:
            reason = []
            if not _has_embeddings: reason.append("missing 'embeddings' key")
            if not _has_codes:      reason.append("barthel_codes empty")
            print(f"  Cache invalid ({'; '.join(reason)}) — deleting and retraining.")
            EMBEDDINGS_CACHE.unlink()
        del _d
    except Exception as e:
        print(f"  Cache unreadable ({e}) — deleting and retraining.")
        EMBEDDINGS_CACHE.unlink()
else:
    print("  No cache found — will train (auto-resumes from latest checkpoint if any).")

# ── Training ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 1: Training Zone A autoencoder")
print("=" * 60)

if CACHE_VALID:
    print("  Skipped (valid cache already present).")
    if not CHECKPOINT.exists():
        CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
elif CHECKPOINT.exists():
    print("zone_a_train already complete — skipping")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/train_autoencoder.py",
            f"paths.corpus_dir={DATA_DIR}/corpus",
            f"paths.glyphs_dir={DATA_DIR}/glyphs",
            f"paths.catalog_dir={DATA_DIR}/catalog",
            f"paths.outputs_dir={REPO_DIR}/outputs",
            f"paths.checkpoints_dir={str(ckpt_dst)}",
            f"paths.embeddings_cache={EMBEDDINGS_CACHE}",
            "zone_a.autoencoder.num_epochs=50",
            "zone_a.autoencoder.batch_size=64",
            # "auto" globs checkpoints_dir for the latest .pt and resumes.
            # Change to "none" to force a full retrain from scratch.
            "zone_a.autoencoder.resume_checkpoint=auto",
            "hydra.job.chdir=false",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=STEP_TIMEOUT)
        if proc.returncode == 0:
            CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
            print(f"\nDone. Embeddings cache at: {EMBEDDINGS_CACHE}")
        else:
            print(f"WARNING: zone_a_train failed (exit {proc.returncode}) — continuing")
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"WARNING: zone_a_train timed out after {STEP_TIMEOUT // 60} min — skipping")

# ── Verify + save to Drive ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("VERIFICATION + SAVING TO DRIVE")
print("=" * 60)

ckpts = sorted(ckpt_dst.glob("autoencoder_epoch*.pt"))
for f in ckpts:
    dest = Path(CHECKPOINTS_DIR) / f.name
    shutil.copy(f, dest)
    print(f"  ✓ {f.name}  {f.stat().st_size/1024/1024:.1f} MB → Drive")
if not ckpts:
    print("  ✗ No checkpoints found")

if EMBEDDINGS_CACHE.exists():
    shutil.copy(EMBEDDINGS_CACHE, Path(CHECKPOINTS_DIR) / "embeddings_cache.pt")
    print(f"  ✓ embeddings_cache.pt  {EMBEDDINGS_CACHE.stat().st_size/1024/1024:.1f} MB → Drive")
    print("\nZone A Cell 2 is ready to run.")
else:
    print("  ✗ embeddings_cache.pt NOT FOUND — check logs above")

In [ ]:
# ── Cell A: Zone A with DINOv2 backbone ──────────────────────────────────────
# Only runs if zone_a.backbone = "dinov2" in config.yaml
# Much faster than custom CNN: backbone is frozen, only projection head trains
# Expected time: ~5 min on T4 GPU vs ~20 min for custom CNN

import subprocess, sys, os
from datetime import datetime, timezone
from pathlib import Path
STEP_TIMEOUT = 1800  # seconds; kill and skip if exceeded
REPO_DIR = "/content/repo"  # colab path
CHECKPOINT = Path(f"{REPO_DIR}/outputs/checkpoints/zone_a_dinov2.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if RUN_MODE not in ("experimental", "redrongo"):
    print("Skipping DINOv2 cell (set RUN_MODE='experimental' to enable)")
else:
    os.chdir(REPO_DIR)

    import yaml
    with open('conf/config.yaml') as f:
        cfg = yaml.safe_load(f)
    backbone = cfg.get('zone_a', {}).get('backbone', 'custom')
    print(f"Zone A backbone: {backbone}")

    if backbone == 'dinov2':
        if CHECKPOINT.exists():
            print("zone_a_dinov2 already complete — skipping")
        else:
            proc = subprocess.Popen(
                [sys.executable, "scripts/train_autoencoder.py",
                 "zone_a.backbone=dinov2",
                 "hydra.job.chdir=false"],
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
            )
            try:
                for line in proc.stdout:
                    print(line, end="")
                proc.wait(timeout=STEP_TIMEOUT)
                if proc.returncode == 0:
                    CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
                else:
                    print(f"WARNING: zone_a_dinov2 failed (exit {proc.returncode}) — continuing")
            except subprocess.TimeoutExpired:
                proc.kill()
                print(f"WARNING: zone_a_dinov2 timed out after {STEP_TIMEOUT // 60} min — skipping")
    else:
        print("Custom CNN backbone configured — skipping DINOv2 cell")


In [ ]:
# ── Cell B: Cross-script similarity (Hevesy hypothesis test) ─────────────────
# Tests whether rongorongo and Indus Valley script share non-trivial
# visual overlap using shared DINOv2 embeddings.
# Requires: Indus Valley glyph images (auto-downloaded by fetch_indus_glyphs.py)

import subprocess, sys, os, json
from datetime import datetime, timezone
from pathlib import Path

STEP_TIMEOUT = 1800  # seconds; kill and skip if exceeded
REPO_DIR = "/content/repo"
CHECKPOINT = Path(f"{REPO_DIR}/outputs/checkpoints/cross_script_similarity.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if RUN_MODE not in ("experimental", "redrongo"):
    print("Skipping cross-script cell (set RUN_MODE='experimental' to enable)")
else:
    os.chdir(REPO_DIR)

    # ── Render rongorongo reference glyphs from SVGs ─────────────────────────────
    # data/barthel_ref/ holds one 128×128 PNG per unique Barthel code, used as
    # the rongo-dir for the DINOv2 cross-script comparison.
    # If the directory is missing or empty we render it now from the SVG catalog
    # (data/glyphs/svg/catalog.json + individual SVG files scraped from
    # kohaumotu.org — these are always present in the data zip).

    _rongo_dir = Path(f"{REPO_DIR}/data/barthel_ref")
    _svg_base   = Path(f"{REPO_DIR}/data/glyphs/svg")
    _catalog    = _svg_base / "catalog.json"

    _needs_render = not _rongo_dir.exists() or not any(_rongo_dir.glob("*.png"))

    if _needs_render and _catalog.exists():
        import cairosvg
        from PIL import Image
        import io, numpy as np

        _rongo_dir.mkdir(parents=True, exist_ok=True)
        _records = json.loads(_catalog.read_text()).get("records", [])

        # Keep only the first SVG occurrence per Barthel code.
        _seen: dict = {}
        for rec in _records:
            code = rec.get("barthel_code")
            svg_rel = rec.get("svg_path", "")
            if code and svg_rel and code not in _seen:
                full = _svg_base / svg_rel.replace("svg/", "")
                if full.exists():
                    _seen[code] = full

        _written = 0
        for code, svg_path in _seen.items():
            try:
                # SVGs from kohaumotu.org have no explicit fill attribute;
                # inject fill:#000000 so cairosvg renders visible black strokes.
                svg_text = svg_path.read_text(encoding="utf-8")
                svg_text = svg_text.replace("<path ", '<path style="fill:#000000;" ')
                png_bytes = cairosvg.svg2png(
                    bytestring=svg_text.encode(),
                    output_width=128, output_height=128,
                    background_color="white",
                )
                img = Image.open(io.BytesIO(png_bytes)).convert("L")  # greyscale
                out_name = "".join(c if c.isalnum() or c in ".-_!?:" else "_" for c in code)
                img.save(_rongo_dir / f"{out_name}.png")  # L-mode greyscale PNG
                _written += 1
            except Exception as _e:
                pass  # skip malformed SVGs silently

        print(f"  Rendered {_written} rongorongo reference glyphs → {_rongo_dir}")
    elif _needs_render:
        print(f"  WARNING: SVG catalog not found at {_catalog}")
        print(f"  Cross-script similarity will fail. Run Setup Cell 3 first.")
    else:
        _n = len(list(_rongo_dir.glob("*.png")))
        print(f"  Rongorongo reference glyphs already present ({_n} PNGs) — skipping render.")

    for fetch_script in ["scripts/fetch_indus_glyphs.py",
                         "scripts/render_linearb_glyphs.py"]:
        if Path(fetch_script).exists():
            proc = subprocess.run(
                [sys.executable, fetch_script],
                capture_output=True, text=True
            )
            print(proc.stdout[-500:] if proc.stdout else "")

    if CHECKPOINT.exists():
        print("cross_script_similarity already complete — skipping")
    else:
        proc = subprocess.Popen(
            [
                sys.executable, "scripts/cross_script_similarity.py",
                "--rongo-dir",   f"{REPO_DIR}/data/barthel_ref",  # canonical sign images (1345 PNGs, always present)
                "--indus-dir",   f"{REPO_DIR}/data/glyphs/indus",
                "--control-dir", f"{REPO_DIR}/data/glyphs/linear_b",
                "--output",      f"{REPO_DIR}/outputs/analysis/cross_script_similarity.json",
                "--report",      f"{REPO_DIR}/outputs/analysis/cross_script_similarity_report.html",
                "--top-k",       "50",
            ],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        try:
            for line in proc.stdout:
                print(line, end="")
            proc.wait(timeout=STEP_TIMEOUT)
            if proc.returncode == 0:
                CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
            else:
                print(f"WARNING: cross_script_similarity failed (exit {proc.returncode}) — continuing")
        except subprocess.TimeoutExpired:
            proc.kill()
            print(f"WARNING: cross_script_similarity timed out after {STEP_TIMEOUT // 60} min — skipping")

    result_path = Path(f"{REPO_DIR}/outputs/analysis/cross_script_similarity.json")
    if result_path.exists():
        r = json.loads(result_path.read_text())
        print(f"\n── Cross-Script Similarity Summary ────────────────────────")
        print(f"  Rongorongo glyphs:         {r['n_rongo']}")
        print(f"  Indus Valley signs:        {r['n_indus']}")
        print(f"  KS p-value vs control A:   {r['ks_test_indus_vs_control_a']['p_value']:.4f}")
        print(f"  KS p-value vs control B:   {r['ks_test_indus_vs_control_b']['p_value']:.4f}")
        print(f"  Hevesy pairs recovered:    {r['hevesy_pairs_recovered']}/40")
        print(f"  Hevesy recovery rate:      {r['hevesy_recovery_rate']:.1%}")
        if r['ks_test_indus_vs_control_a']['p_value'] < 0.05:
            print(f"\n  *** SIGNIFICANT: Rongorongo-Indus similarity exceeds control ***")
            print(f"  Hevesy hypothesis has computational support.")
        else:
            print(f"\n  Result: no significant excess similarity over control.")
            print(f"  Hevesy hypothesis not supported at p < 0.05.")


In [ ]:
# ── Zone A Cell 2: Analyse embeddings — UMAP + HDBSCAN + reports ─────────────
# Produces: umap_embeddings.png, cluster_vs_barthel.csv/.json,
#           divergence_report.html, compound_candidates.json,
#           compound_report.html
# Only run after Zone A Cell 1 prints "Zone A Cell 2 is ready to run."

import subprocess, sys, os, json
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import Image, display

STEP_TIMEOUT = 1800  # seconds; kill and skip if exceeded
REPO_DIR         = "/content/repo"
CHECKPOINTS_DIR  = "/content/drive/MyDrive/hackingrongo_checkpoints"
EMBEDDINGS_CACHE = f"{CHECKPOINTS_DIR}/embeddings_cache.pt"
DATA_DIR         = f"{REPO_DIR}/data"
ANALYSIS_OUT     = f"{REPO_DIR}/outputs/analysis"
CHECKPOINT       = Path(f"{REPO_DIR}/outputs/checkpoints/analyze_embeddings.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
Path(ANALYSIS_OUT).mkdir(parents=True, exist_ok=True)

if not Path(EMBEDDINGS_CACHE).exists():
    raise FileNotFoundError("embeddings_cache.pt not found. Run Zone A Cell 1 first.")
print(f"Cache: {Path(EMBEDDINGS_CACHE).stat().st_size/1024/1024:.1f} MB  ✓")

# ── Step 1: UMAP + HDBSCAN + divergence report ────────────────────────────────
print("\n" + "=" * 60)
print("STEP 1: Embedding analysis + divergence report")
print("=" * 60)

if CHECKPOINT.exists():
    print("analyze_embeddings already complete — skipping")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/analyze_embeddings.py",
            f"paths.embeddings_cache={EMBEDDINGS_CACHE}",
            f"paths.outputs_dir={REPO_DIR}/outputs",
            "hydra.job.chdir=false",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=STEP_TIMEOUT)
        if proc.returncode == 0:
            CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
        else:
            print(f"WARNING: analyze_embeddings failed (exit {proc.returncode}) — continuing")
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"WARNING: analyze_embeddings timed out after {STEP_TIMEOUT // 60} min — skipping")

# ── Step 2: Compound detector — validation run ────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Compound detector — P@k validation on known compounds")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.compound_detector",
        "--analysis-dir", "outputs/analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output", "outputs/analysis/compound_validation.json",
        "--include-known", "--min-methods", "2", "--min-confidence", "0.25",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    print(f"WARNING: Validation run failed (exit {proc.returncode}) — continuing")

val_path = Path(ANALYSIS_OUT) / "compound_validation.json"
if val_path.exists():
    val   = json.loads(val_path.read_text())
    cands = val.get("candidates", [])
    print("\nPrecision@k on known compounds:")
    for k in [5, 10, 20]:
        top     = cands[:k]
        n_known = sum(1 for c in top if c.get("is_known_compound", False))
        p       = n_known / max(len(top), 1)
        bar     = "█" * round(p * 20) + "░" * (20 - round(p * 20))
        print(f"  P@{k:2d}: {p:.2f}  {bar}  ({n_known}/{len(top)})")

# ── Step 3: Compound detector — novel candidates ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Compound detector — novel candidates")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.compound_detector",
        "--analysis-dir", "outputs/analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output", "outputs/analysis/compound_candidates.json",
        "--min-methods", "2", "--min-confidence", "0.25",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    print(f"WARNING: Detection failed (exit {proc.returncode}) — continuing")

# ── Step 4: Compound report (scholarly comparison always; UMAP candidates if available) ──
print("\n" + "=" * 60)
print("STEP 4: Compound report")
print("=" * 60)

compound_args = [
    sys.executable, "-m", "hackingrongo.results.compound_report",
    "--svg-catalog", f"{DATA_DIR}/glyphs/svg/catalog.json",
    "--corpus-dir",  f"{DATA_DIR}/corpus",
    "--output",      "outputs/analysis/compound_report.html",
    "--max-candidates", "50",
]
# Include UMAP-based candidates if available (Zone A must have run first)
if (Path(ANALYSIS_OUT) / "compound_candidates.json").exists():
    compound_args += ["--candidates", "outputs/analysis/compound_candidates.json"]
else:
    print("  No compound_candidates.json — generating scholarly comparison section only")

proc = subprocess.Popen(
    compound_args,
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    print(f"WARNING: Compound report failed (exit {proc.returncode}) — continuing")

# ── Step 5: Parallel passage cross-reference ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Parallel passage cross-reference")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "scripts/cross_reference_parallels.py",
        "--input",     f"{DATA_DIR}/parallels/horley_parallels.csv",
        "--corpus",    f"{DATA_DIR}/corpus",
        "--config",    f"{REPO_DIR}/conf/config.yaml",
        "--tablets",   f"{DATA_DIR}/metadata/tablets.json",
        "--output",    f"{DATA_DIR}/parallels/parallel_variants_auto.json",
        "--threshold", "1",
        "--verbose",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    print(f"WARNING: Cross-reference failed (exit {proc.returncode}) — continuing")

# ── Step 6: Diachronic passage report ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: Diachronic passage report")
print("=" * 60)

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
if variants_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.passage_report",
            "--input",  str(variants_path),
            "--output", f"{REPO_DIR}/outputs/analysis/passage_reports",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    try:
        proc.wait(timeout=STEP_TIMEOUT)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
        print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
        proc.returncode = -1
    if proc.returncode != 0:
        print(f"WARNING: Passage report failed (exit {proc.returncode}) — continuing")
else:
    print("parallel_variants_auto.json not found — skipping passage report")
    
# ── Step 7: Display cluster summary + UMAP ───────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: Results")
print("=" * 60)

json_path = Path(ANALYSIS_OUT) / "cluster_vs_barthel.json"
if json_path.exists():
    result = json.loads(json_path.read_text())
    print(f"  Embeddings : {result['n_embeddings']}")
    print(f"  Clusters   : {result['n_clusters']}")
    print(f"  Noise pts  : {result['n_noise_points']}")
    ari = result.get("adjusted_rand_index")
    if ari is not None:
        print(f"  ARI        : {ari:.4f}  ({result.get('interpretation', '')})")
    print("\nTop 5 clusters by size:")
    clusters = {k: v for k, v in result.get("clusters", {}).items() if k != "noise"}
    for cid, info in sorted(clusters.items(), key=lambda x: -x[1]["size"])[:5]:
        top_codes = list(info["barthel_codes"].keys())[:4]
        top_fams  = list(info["barthel_families"].keys())[:2]
        print(f"  Cluster {cid}: {info['size']} glyphs  codes={top_codes}  families={top_fams}")

plot_path = Path(ANALYSIS_OUT) / "umap_embeddings.png"
if plot_path.exists():
    print("\nUMAP scatter:")
    display(Image(filename=str(plot_path)))

# ── Step 8: Astronomical glyph candidate analysis ────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: Astronomical glyph candidate analysis")
print("=" * 60)

Path(f"{REPO_DIR}/outputs/zone_b").mkdir(parents=True, exist_ok=True)
proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.astronomical_analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output",     f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    print(f"WARNING: Astronomical analysis failed (exit {proc.returncode}) — continuing")

# Print top candidates inline
astro_path = Path(f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json")
if astro_path.exists():
    try:
        astro_data  = json.loads(astro_path.read_text())
        candidates  = astro_data.get("candidates", [])
        print(f"\n  Candidates flagged: {len(candidates)}")
        if candidates:
            print(f"  {'Code':<8} {'Score':>6}  {'Methods':>7}  Details")
            print(f"  {'-'*8} {'-'*6}  {'-'*7}  {'-'*35}")
            for c in candidates[:10]:
                code   = c.get("barthel_code", "?")
                score  = c.get("overall_score", c.get("consensus_confidence", 0))
                n      = c.get("n_methods_flagged", c.get("n_methods_agreeing", 0))
                entry  = c.get("dietrich_entry")          # may be None
                if isinstance(entry, dict):
                    desc = entry.get("proposed_referent", "—")
                elif entry is not None:
                    desc = str(entry)
                else:
                    desc = "bird-headed / no Dietrich entry"
                print(f"  {code:<8} {score:>6.3f}  {n:>7}  {desc}")
    except Exception as e:
        print(f"  WARNING: Could not display candidates inline: {e}")
        print(f"  (astronomical_candidates.json was written — Step 9 will render it properly)")
        
# ── Step 9: Astronomical HTML report ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: Astronomical report")
print("=" * 60)

if astro_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.astronomical_report",
            "--candidates", str(astro_path),
            "--svg-catalog", f"{DATA_DIR}/glyphs/svg/catalog.json",
            "--output",     f"{REPO_DIR}/outputs/analysis/astronomical_report.html",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    try:
        proc.wait(timeout=STEP_TIMEOUT)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
        print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
        proc.returncode = -1
    if proc.returncode != 0:
        print(f"WARNING: Astronomical report failed (exit {proc.returncode}) — continuing")
else:
    print("astronomical_candidates.json not found — skipping report")

# ── Step 10: Train sequence model (for glyph reconstruction) ─────────────────
print("\n" + "=" * 60)
print("STEP 10: Train sequence model")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/train_sequence_model.py",
        "--output",     f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
        "--order",      "3",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    print(f"WARNING: Sequence model training failed — continuing")
else:
    # Run reconstruction on Tablet F (81 glyphs, several uncertain readings)
    proc2 = subprocess.Popen(
        [
            sys.executable, "scripts/complete_sequence.py",
            "--tablet",  "F",
            "--model",   f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
            "--out",     f"{REPO_DIR}/outputs/zone_b/tablet_f_reconstruction.json",
            "--top-k",   "5",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc2.stdout:
        print(line, end="")
    try:
        proc2.wait(timeout=STEP_TIMEOUT)
    except subprocess.TimeoutExpired:
        proc2.kill()
        proc2.wait()
        print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
        proc2.returncode = -1

# ── Step 8b: Quantum p_good measurement ───────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8b: Quantum hardness analysis (p_good measurement)")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/measure_pgood.py",
        "--corpus-dir",      f"{DATA_DIR}/corpus",
        "--lm-dir",          f"{REPO_DIR}/data/language_models",
        "--n-samples",       "10000",
        "--thresholds",      "0.90,0.95,0.99",
        "--mcmc-iterations", "5000",
        "--output",          f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    print(f"WARNING: p_good measurement failed — continuing")

pgood_path = Path(f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json")
if pgood_path.exists():
    pg = json.loads(pgood_path.read_text())
    print(f"\n  ── Quantum hardness summary ──────────────────────────────")
    print(f"  Score distribution: mean={pg['score_distribution']['mean']:.1f} "
          f"std={pg['score_distribution']['std']:.1f}")
    for t in pg['thresholds']:
        print(f"  τ={t['tau']:.2f}: p_good={t['p_good']:.2e}  "
              f"Grover={t['grover_oracle_calls']:,} calls  "
              f"Classical={t['classical_random_calls']:,}  "
              f"Speedup={t['quantum_speedup_ratio']:.1f}x")

print("\nZone C Cell 1 is ready to run.")

In [ ]:
# ── Cell 6d: Zone C fusion layer training (step 4k) ─────────────────────
# Trains the FusionLayer that combines Zone A embeddings + Zone B priors.
# Requires: Zone A Cell 1 (embeddings_cache.pt) and Zone A Cell 2 (compound_candidates.json).
# Output saved to Drive: zone_c/fusion_checkpoint.pt
#
# Safe to skip — Zone C Cell 2 (MCMC) falls back to raw embeddings if the
# checkpoint is absent.

import subprocess, sys, os, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
os.chdir(REPO_DIR)

EMBEDDINGS = Path(f"{REPO_DIR}/outputs/embeddings_cache.pt")
COMPOUNDS  = Path(f"{REPO_DIR}/outputs/analysis/compound_candidates.json")
FUSION_OUT = Path(f"{REPO_DIR}/outputs/zone_c/fusion_checkpoint.pt")

if not EMBEDDINGS.exists():
    print("embeddings_cache.pt not found — run Zone A Cell 1 first.")
elif not COMPOUNDS.exists():
    print("compound_candidates.json not found — run Zone A Cell 2 first.")
else:
    FUSION_OUT.parent.mkdir(parents=True, exist_ok=True)
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/train_fusion.py",
            "--embeddings", str(EMBEDDINGS),
            "--compounds",  str(COMPOUNDS),
            "--corpus-dir", f"{DATA_DIR}/corpus",
            "--output",     str(FUSION_OUT),
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: train_fusion failed (exit {proc.returncode}) — continuing")
    elif FUSION_OUT.exists():
        shutil.copy(FUSION_OUT, Path(CHECKPOINTS_DIR) / "fusion_checkpoint.pt")
        print(f"  ✓ fusion_checkpoint.pt ({FUSION_OUT.stat().st_size/1024:.0f} KB) saved to Drive")

In [ ]:
# ── Zone B Cell 1: Reading order tests (basic) ─────────────────────────────────
# Tests 1 & 2 use the sequence model from Zone A Cell 2 (Step 10).
# Tests 3 & 4 (line-boundary entropy + recto/verso ordering) run corpus-only.
# Test 4 resolves Pozdniakov's 1958 open question on reading-side ordering.
#
# Output saved to Drive: reading_order_results.json

import subprocess, sys, os, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
os.chdir(REPO_DIR)

RESULTS_JSON = Path(f"{REPO_DIR}/outputs/reading_order_results.json")
CORPUS_DIR   = f"{REPO_DIR}/data/corpus"
MODEL_PATH   = f"{REPO_DIR}/outputs/zone_b/sequence_model.json"

proc = subprocess.Popen(
    [sys.executable, "scripts/reading_order_tests.py",
     "--corpus", CORPUS_DIR,
     "--model",  MODEL_PATH,
     "--tests",  "1", "2", "3", "4",
     "--output", str(RESULTS_JSON)],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: reading_order_tests failed (exit {proc.returncode})")

# ── Save to Drive ─────────────────────────────────────────────────────────────
if RESULTS_JSON.exists():
    shutil.copy(RESULTS_JSON, f"{CHECKPOINTS_DIR}/reading_order_results.json")
    print(f"  ✓ reading_order_results.json saved to Drive")

In [ ]:
# ── Zone B Cell 2: Build language models and parallel passages ─────────────────
# Run once after Zone A Cell 2. Takes ~5 minutes.
# Language models are required for Zone C Cell 2.
# Parallel passage cross-reference finds multi-tablet attestations.
#
# Outputs saved to Drive:
#   parallel_variants_auto.json   ← 13 multi-tablet passages found
#   pre_contact_lm.json           ← Thomson 1891 + Roussel 1908 (~1345 forms)
#   post_contact_lm.json          ← Barthel/Blixen/Fischer + IDS (~2754 forms)
#   smoothing_lm.json             ← Hawaiian Corpus Project (~56K types)

import subprocess, sys, json, shutil, os
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"

os.chdir(REPO_DIR)

# ── Step 1: Fetch IDS Rapa Nui vocabulary ─────────────────────────────────────
print("=" * 60)
print("STEP 1: Fetch IDS Rapa Nui vocabulary (Thomson + Roussel)")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/parse_ids.py",
        "--stratify",
        "--cache-dir", f"{DATA_DIR}/cache",
        "--data-dir",  f"{DATA_DIR}/polynesian_texts",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: parse_ids.py failed (exit {proc.returncode}) — continuing")

# ── Step 2: Fetch ABVD cognate neighbours ─────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Fetch ABVD East Polynesian cognate neighbours")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/fetch_abvd_corpus.py",
        "--with-cognates",
        "--cache-dir", f"{DATA_DIR}/cache",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: fetch_abvd_corpus.py failed — continuing")

# ── Step 2b: Fetch Hawaiian smoothing corpus ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2b: Fetch Hawaiian smoothing corpus")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/fetch_hawaiian_corpus.py",
        "--cache-dir", f"{DATA_DIR}/cache",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: fetch_hawaiian_corpus.py failed — continuing")

# ── Step 3: Build language models ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Build stratified language models")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/build_language_models.py",
        "hydra.job.chdir=false",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"build_language_models.py failed (exit {proc.returncode})")

print("\n── Language model verification ──────────────────────────────")
lm_dir = Path(f"{REPO_DIR}/data/language_models")
lm_ok = True
for lm_name in ["pre_contact_lm", "post_contact_lm", "smoothing_lm"]:
    p = lm_dir / f"{lm_name}.json"
    if p.exists():
        vocab = len(json.loads(p.read_text()).get("vocab", []))
        status = "✓" if vocab >= 50 else "✗ TOO SPARSE"
        print(f"  {status} {lm_name}: vocab={vocab} types")
        if vocab < 50:
            lm_ok = False
    else:
        print(f"  ✗ {lm_name}.json MISSING")
        lm_ok = False

if not lm_ok:
    print("\nWARNING: LMs too sparse — Zone C will produce uninformative results")
else:
    print("\nLMs ready for Zone C.")

# ── Step 4: Parallel passage cross-reference ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Parallel passage cross-reference")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/cross_reference_parallels.py",
        "--input",     f"{DATA_DIR}/parallels/horley_parallels.csv",
        "--corpus",    f"{DATA_DIR}/corpus",
        "--config",    f"{REPO_DIR}/conf/config.yaml",
        "--tablets",   f"{DATA_DIR}/metadata/tablets.json",
        "--output",    f"{DATA_DIR}/parallels/parallel_variants_auto.json",
        "--threshold", "1",
        "--verbose",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Cross-reference failed (exit {proc.returncode}) — continuing")

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
if variants_path.exists():
    v = json.loads(variants_path.read_text())
    n_multi = sum(1 for p in v.get("passages", []) if p.get("n_tablets", 0) >= 2)
    print(f"\n  ✓ parallel_variants_auto.json: {n_multi} multi-tablet passages found")
else:
    print("  ✗ parallel_variants_auto.json not generated")

# ── Step 5: Passage report ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Diachronic passage report")
print("=" * 60)
if variants_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.passage_report",
            "--input",        str(variants_path),
            "--output",       f"{REPO_DIR}/outputs/analysis/passage_reports",
            "--filter-score", "0.0",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Passage report failed (exit {proc.returncode}) — continuing")
else:
    print("Skipped — parallel_variants_auto.json not found")

# ── Save to Drive ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Saving to Drive")
print("=" * 60)

for f in sorted(lm_dir.glob("*.json")):
    shutil.copy(f, Path(CHECKPOINTS_DIR) / f.name)
    print(f"  ✓ {f.name}")

if variants_path.exists():
    shutil.copy(variants_path, Path(CHECKPOINTS_DIR) / "parallel_variants_auto.json")
    print(f"  ✓ parallel_variants_auto.json")

passage_dir = Path(f"{REPO_DIR}/outputs/analysis/passage_reports")
if passage_dir.exists():
    n = 0
    for f in passage_dir.iterdir():
        if f.is_file() and f.suffix == ".html":
            shutil.copy(f, Path(CHECKPOINTS_DIR) / f.name)
            n += 1
    print(f"  ✓ passage_reports/ ({n} HTML files)")

print("\nZone C Cell 2 (MCMC decipherment) is ready to run.")

In [ ]:
# ── Zone B Cell 3: Reading order tests (with HTML report) ─────────────────────
# Four entropy tests for rongorongo transcription direction and recto/verso order.
# Tests 3 & 4 need only the corpus — they run immediately.
# Tests 1 & 2 need the sequence model from Zone A Cell 2 — skipped automatically if absent.
#
# Test 1: Conditional entropy asymmetry  H_forward vs H_reverse
# Test 2: N-gram model perplexity asymmetry (trained model, forward vs reversed)
# Test 3: Line-boundary entropy — are line breaks real compositional units?
# Test 4: Recto/verso ordering — resolves Pozdniakov's question from 1958
#
# Outputs written to Drive:
#   reading_order_results.json       ← structured test results (JSON)
#   reading_order_report.html        ← scholar-facing HTML report

import os, subprocess, sys, json, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"

os.chdir(REPO_DIR)

RESULTS_JSON = Path(f"{REPO_DIR}/outputs/reading_order_results.json")
REPORT_HTML  = Path(f"{REPO_DIR}/outputs/analysis/reading_order_report.html")
CORPUS_DIR   = Path(f"{DATA_DIR}/corpus")
MODEL_PATH   = Path(f"{REPO_DIR}/outputs/zone_b/sequence_model.json")

if not CORPUS_DIR.exists():
    raise RuntimeError(f"Corpus directory not found: {CORPUS_DIR}\nRun Cell 3 first.")

# ── Tests 3 & 4 (no model required) ──────────────────────────────────────────
print("=" * 60)
print("READING ORDER TESTS 3 & 4 (corpus only)")
print("=" * 60)
subprocess.check_call(
    [
        sys.executable, "scripts/reading_order_tests.py",
        "--corpus", str(CORPUS_DIR),
        "--tests",  "3", "4",
        "--output", str(RESULTS_JSON),
    ],
    cwd=REPO_DIR,
)

# ── Tests 1 & 2 (sequence model required) ────────────────────────────────────
if MODEL_PATH.exists():
    print("\n" + "=" * 60)
    print("READING ORDER TESTS 1 & 2 (sequence model found)")
    print("=" * 60)
    subprocess.check_call(
        [
            sys.executable, "scripts/reading_order_tests.py",
            "--corpus", str(CORPUS_DIR),
            "--model",  str(MODEL_PATH),
            "--tests",  "1", "2", "3", "4",
            "--output", str(RESULTS_JSON),
        ],
        cwd=REPO_DIR,
    )
else:
    print(f"\n  Sequence model not found at {MODEL_PATH}")
    print("  Tests 1 & 2 skipped. Run Zone A Cell 1 to train the model, then re-run this cell.")

# ── Generate HTML report ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("GENERATING READING ORDER REPORT")
print("=" * 60)
subprocess.check_call(
    [
        sys.executable, "-m", "hackingrongo.results.reading_order_report",
        "--input",  str(RESULTS_JSON),
        "--output", str(REPORT_HTML),
    ],
    cwd=REPO_DIR,
)

# ── Display key results ───────────────────────────────────────────────────────
if RESULTS_JSON.exists():
    results = json.loads(RESULTS_JSON.read_text())
    t4 = results.get("test4", {})
    t3 = results.get("test3", {})
    if t4:
        print("\n" + "=" * 60)
        print("TEST 4  RECTO / VERSO ORDERING")
        print("=" * 60)
        print(f"  {t4.get('verdict_text', '(no verdict)')}")
        ppl_ab2 = t4.get("ppl_ab_bigram");  ppl_ba2 = t4.get("ppl_ba_bigram")
        ppl_ab3 = t4.get("ppl_ab_trigram"); ppl_ba3 = t4.get("ppl_ba_trigram")
        if all(v is not None for v in [ppl_ab2, ppl_ba2, ppl_ab3, ppl_ba3]):
            print(f"  Bigram  LOO-PPL   a→b : {ppl_ab2:.2f}   b→a : {ppl_ba2:.2f}")
            print(f"  Trigram LOO-PPL   a→b : {ppl_ab3:.2f}   b→a : {ppl_ba3:.2f}")
    if t3:
        print("\nTEST 3  LINE BOUNDARY ENTROPY")
        print(f"  {t3.get('verdict_text', '(no verdict)')}")


# ── Generate tablet spectrum scores ──────────────────────────────────────────
print("\n" + "=" * 60)
print("GENERATING TABLET SPECTRUM SCORES")
print("=" * 60)

SPECTRUM_JSON = Path(f"{REPO_DIR}/outputs/analysis/spectrum_scores.json")

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.spectrum_analyzer",
        "--corpus-dir", str(CORPUS_DIR),
        "--metadata",   f"{DATA_DIR}/metadata/tablets.json",
        "--output",     str(SPECTRUM_JSON),
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Spectrum analyzer failed (exit {proc.returncode}) — continuing")
elif SPECTRUM_JSON.exists():
    sp_data = json.loads(SPECTRUM_JSON.read_text())
    tablets_sp = sp_data.get("tablets", {})
    sorted_sp  = sorted(tablets_sp.items(), key=lambda x: -x[1].get("spectrum_score", 0))
    print(f"\n  Tablets: {sp_data.get('n_tablets','?')}  Reliable: {sp_data.get('n_reliable','?')}")
    print("  Top 5 most logographic:")
    for tid, feat in sorted_sp[:5]:
        score = feat.get("spectrum_score", 0)
        bar   = "█" * round(score * 20) + "░" * (20 - round(score * 20))
        print(f"    {tid}  {feat.get('tablet_name',''):<22} {score:.3f}  {bar}")

# ── Generate combined entropy + reading direction + spectrum report ────────────
print("\n" + "=" * 60)
print("GENERATING COMBINED ENTROPY REPORT (Parts I + II + III)")
print("=" * 60)

COMBINED_REPORT  = Path(f"{REPO_DIR}/outputs/analysis/entropy_report.html")
SENSITIVITY_JSON = Path(f"{REPO_DIR}/outputs/sensitivity_analysis.json")

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.results.entropy_report",
        "--sensitivity",   str(SENSITIVITY_JSON),
        "--zipf",          f"{REPO_DIR}/outputs/analysis/zipf_analysis.json",
        "--boustrophedon", f"{REPO_DIR}/outputs/analysis/boustrophedon_ic.json",
        "--reading-order", str(RESULTS_JSON),
        "--spectrum",      str(SPECTRUM_JSON),
        "--metadata",      f"{DATA_DIR}/metadata/tablets.json",
        "--corpus-dir",    str(CORPUS_DIR),
        "--svg-catalog",   f"{DATA_DIR}/glyphs/svg/catalog.json",
        "--output",        str(COMBINED_REPORT),
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Combined entropy report failed (exit {proc.returncode}) — continuing")
elif COMBINED_REPORT.exists():
    print(f"\n  ✓ entropy_report.html  ({COMBINED_REPORT.stat().st_size/1024:.0f} KB)")
    print("    Part I: Entropy & Information Theory")
    print("    Part II: Reading Direction (Tests 1-4)")
    print("    Part III: Tablet Spectrum (logographic ↔ syllabic)")

# ── Copy new outputs to Drive ─────────────────────────────────────────────────
for src, name in [
    (SPECTRUM_JSON,  "spectrum_scores.json"),
    (COMBINED_REPORT, "entropy_report.html"),
]:
    if src.exists():
        shutil.copy(src, f"{CHECKPOINTS_DIR}/{name}")
        print(f"  ✓ {name}  ({src.stat().st_size / 1024:.0f} KB)")
    else:
        print(f"  — {name}  (not generated)")

# ── Copy to Drive ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Copying to Drive")
print("=" * 60)
for src, name in [
    (RESULTS_JSON, "reading_order_results.json"),
    (REPORT_HTML,  "reading_order_report.html"),
]:
    if src.exists():
        shutil.copy(src, f"{CHECKPOINTS_DIR}/{name}")
        print(f"  ✓ {name}  ({src.stat().st_size / 1024:.0f} KB)")
    else:
        print(f"  — {name}  (not generated)")

In [ ]:
# ── Zone B Cell 4: Parallel passage ranking and download ──────────────────────
# Shows all parallel passages ranked by attestation count (number of tablets
# they appear on). The top passage is the best target for focused decipherment
# in Zone C Cell 2 — it has the most cross-tablet variant evidence to constrain MCMC.
#
# Outputs:
#   passage_reports/index.html   ← summary of all passages
#   passage_reports/{pid}.html   ← one file per passage (glyph images + alignment)
#
# Downloads index.html and the top passage directly to your browser.

import json, shutil
from pathlib import Path
from google.colab import files
from IPython.display import HTML, display

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
PASSAGE_REPORTS = f"{REPO_DIR}/outputs/analysis/passage_reports"

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")

if not variants_path.exists():
    print("parallel_variants_auto.json not found — run Zone B Cell 2 first.")
else:
    raw = json.loads(variants_path.read_text())
    passages = raw.get("passages", raw if isinstance(raw, list) else [])

    # Rank by number of distinct tablets, then total variant count.
    ranked = sorted(
        [p for p in passages if isinstance(p, dict)],
        key=lambda p: (p.get("n_tablets", len(p.get("variants", []))), len(p.get("variants", []))),
        reverse=True,
    )

    print(f"{'ID':<10} {'Tablets':>7}  {'Variants':>8}  {'Length':>6}  Attested on")
    print("-" * 70)
    for p in ranked[:20]:
        pid      = p.get("passage_id", p.get("id", "?"))
        n_tab    = p.get("n_tablets", "?")
        n_var    = len(p.get("variants", []))
        canon    = p.get("canonical_form", [])
        n_glyphs = len(canon)
        tab_ids  = sorted({v.get("tablet_id", "?") for v in p.get("variants", [])})
        print(f"  {str(pid):<8} {str(n_tab):>7}  {n_var:>8}  {n_glyphs:>6}  {', '.join(tab_ids)}")

    if ranked:
        top = ranked[0]
        TOP_PASSAGE_ID = str(top.get("passage_id", top.get("id", "")))
        top_tabs = sorted({v.get("tablet_id", "?") for v in top.get("variants", [])})
        print(f"\n→ Top passage: {TOP_PASSAGE_ID}  ({top.get('n_tablets', '?')} tablets: {', '.join(top_tabs)})")
        print(f"  Set FOCUS_PASSAGE = \"{TOP_PASSAGE_ID}\" in Zone C Cell 2 to run focused decipherment.")
    else:
        TOP_PASSAGE_ID = ""
        print("No passages found.")

    # Download passage HTML reports if they were generated.
    report_dir = Path(PASSAGE_REPORTS)
    if report_dir.exists():
        html_files = sorted(report_dir.glob("*.html"))
        print(f"\n{len(html_files)} HTML report(s) in {PASSAGE_REPORTS}")

        # Save to Drive.
        drive_passage = Path(CHECKPOINTS_DIR) / "passage_reports"
        drive_passage.mkdir(parents=True, exist_ok=True)
        for f in html_files:
            shutil.copy(f, drive_passage / f.name)

        # Download index + top passage to browser.
        for fname in ["index.html"] + ([f"{TOP_PASSAGE_ID}.html"] if TOP_PASSAGE_ID else []):
            p = report_dir / fname
            if p.exists():
                files.download(str(p))
                print(f"  ⬇ {fname}")
            else:
                print(f"  — {fname} not found (passage report may not have been generated yet)")
    else:
        print(f"\nPassage reports directory not found: {PASSAGE_REPORTS}")
        print("Run Zone A Cell 2 Step 6 or Zone B Cell 2 Step 5 to generate them.")

In [ ]:
# ── Zone B Cell 5: Rebuild language models (rescue) ──────────────────────────
# Run this cell if Zone B Cell 2 fails at the LM step but corpus data is already
# present. Rebuilds language models without re-running the full pipeline.

import subprocess, sys, os
REPO_DIR = "/content/repo"
os.chdir(REPO_DIR)

proc = subprocess.Popen(
    [sys.executable, "scripts/build_language_models.py", "hydra.job.chdir=false"],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"Exit code: {proc.returncode}")

In [ ]:
# ── Zone B Cell 3: Sequential context embeddings ─────────────────────────────
# Trains a lightweight 2-layer transformer on rongorongo sign context windows
# (contrastive NT-Xent loss), then computes bigram-based sequential entropy
# H(X_{i+1} | X_i = S) for every sign.
#
# Output drives MCMC proposal weights in Zone C:
#   • High entropy → sign appears in many contexts → likely phonemic → explored more
#   • Zero entropy → stereotyped context → structural logogram → explored less
#
# Expected time: ~5–10 min on CPU (transformer is tiny: context window 9, d=64)
#
# Outputs:
#   sequential_embeddings.pt   ← per-sign transformer context embeddings
#   sequential_entropy.json    ← H(context) per Barthel code (nats)

import os, subprocess, sys, json, shutil
from datetime import datetime, timezone
from pathlib import Path

STEP_TIMEOUT = 2400  # 40 min max; sequential embeddings are optional — timeout is warn-and-skip
REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
CORPUS_DIR      = Path(f"{REPO_DIR}/data/corpus")
EMB_OUT         = Path(f"{REPO_DIR}/outputs/sequential_embeddings.pt")
ENT_OUT         = Path(f"{REPO_DIR}/outputs/sequential_entropy.json")
CHECKPOINT      = Path(f"{REPO_DIR}/outputs/checkpoints/sequential_embeddings.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)

if not CORPUS_DIR.exists():
    raise RuntimeError(f"Corpus not found: {CORPUS_DIR} — run Setup Cell 3 first.")

print("=" * 60)
print("ZONE B CELL 3: SEQUENTIAL CONTEXT EMBEDDINGS")
print("=" * 60)

if CHECKPOINT.exists():
    print("sequential_embeddings already complete — skipping")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/train_sequential_embeddings.py",
            "--corpus-dir", str(CORPUS_DIR),
            "--epochs",     "20",
            "--batch-size", "256",
            "--output",     str(EMB_OUT),
            "--entropy-output", str(ENT_OUT),
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=STEP_TIMEOUT)
        if proc.returncode == 0:
            CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
        else:
            print(f"WARNING: sequential_embeddings failed (exit {proc.returncode}) — continuing")
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"WARNING: sequential_embeddings timed out after {STEP_TIMEOUT // 60} min — skipping")

    _seq_timed_out = CHECKPOINT not in [None] and not CHECKPOINT.exists()
    if _seq_timed_out:
        print("\nSkipping gate check and Drive copy — continuing to next cell.")
    else:
        if proc.returncode != 0 and not CHECKPOINT.exists():
            print("WARNING: sequential_embeddings did not complete — gate check skipped")

        # ── Gate check ──────────────────────────────────────────────────────────────
        if not ENT_OUT.exists():
            print("WARNING: sequential_entropy.json not produced — check output above")

    entropy = json.loads(ENT_OUT.read_text())
    sorted_ent = sorted(entropy.items(), key=lambda kv: kv[1])
    n_zero  = sum(1 for _, v in sorted_ent if v == 0.0)
    n_total = len(sorted_ent)

    print(f"\n  ✓ sequential_entropy.json: {n_total} signs")
    print(f"    Zero-entropy (structural) : {n_zero}  ({100*n_zero/max(n_total,1):.0f}%)")
    print(f"    Median entropy            : {sorted_ent[n_total//2][1]:.3f} nats")

    print("\n  Top 10 highest entropy (phonemic candidates):")
    for code, h in sorted_ent[-10:][::-1]:
        bar = "▓" * int(h * 3)
        print(f"    {code:<10s}  H={h:.3f}  {bar}")

    print("\n  Top 10 lowest non-zero entropy (structural candidates):")
    nonzero = [(c, h) for c, h in sorted_ent if h > 0]
    for code, h in nonzero[:10]:
        bar = "░" * int(h * 3) if h > 0 else ""
        print(f"    {code:<10s}  H={h:.3f}  {bar}")

    # ── Copy to Drive ─────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("Copying to Drive")
    print("=" * 60)
    for src_path, name in [
        (ENT_OUT, "sequential_entropy.json"),
    ]:
        if src_path.exists():
            shutil.copy(src_path, f"{CHECKPOINTS_DIR}/{name}")
            print(f"  ✓ {name}  ({src_path.stat().st_size / 1024:.0f} KB)")

    print("\nZone C Cell 1 is ready to run.")


In [ ]:
# ── Zone C Cell 2: MCMC and beam search decipherment ──────────────────────────
# Requires: Zone B Cell 2 (language models built and saved to Drive).
# Runtime: ~10-20 min full corpus; ~3-5 min focused passage (T4 GPU).
#
# Set FOCUS_PASSAGE to the passage ID from Zone B Cell 4 output for targeted
# decipherment (faster convergence, tighter signal). Leave blank for full corpus.

import subprocess, sys, os, json, shutil
from datetime import datetime, timezone
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
STEP_TIMEOUT    = 3600  # seconds; kill and skip if exceeded
CHECKPOINT      = Path(f"{REPO_DIR}/outputs/checkpoints/zone_c_decipherment.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)

# ── Set passage focus (copy from Cell 7c output, or leave blank) ──────────────
FOCUS_PASSAGE = ""   # e.g. "H-6" — leave blank to decode the full corpus

# ── Build command ─────────────────────────────────────────────────────────────
cmd = [
    sys.executable, "scripts/run_decipherment.py",
    "hydra.job.chdir=false",
    f"paths.corpus_dir={DATA_DIR}/corpus",
    f"paths.outputs_dir={REPO_DIR}/outputs",
    f"paths.language_models_dir={DATA_DIR}/language_models",
    f"paths.parallel_variants_json={DATA_DIR}/parallels/parallel_variants_auto.json",
]
if FOCUS_PASSAGE.strip():
    cmd.append(f"--focus-passage={FOCUS_PASSAGE.strip()}")
    print(f"Focused decipherment on passage: {FOCUS_PASSAGE}")
else:
    print("Full-corpus decipherment ...")

# ── Colab-optimized chain settings ────────────────────────────────────────────
# Colab has 2 vCPUs; spawn-based parallel execution runs 2 chains at a time.
# 2 chains × 20k iterations is sufficient for convergence and completes in
# ~10-15 min vs ~60+ min for the default 4 chains × 50k sequential.
# Detect Colab by filesystem (most reliable) before falling back to env vars.
_on_colab = (os.path.exists("/content") or
             "COLAB_RELEASE_TAG" in os.environ or
             "COLAB_GPU" in os.environ or
             "COLAB_BACKEND_VERSION" in os.environ)
if _on_colab:
    cmd += [
        "zone_c.mcmc.num_chains=2",
        "zone_c.mcmc.num_iterations=20000",
        "zone_c.beam_search.beam_width=10",
        "zone_c.beam_search.max_depth=200",
        "zone_c.beam_search.early_stopping_patience=20",
    ]
    print("Colab: overriding to 2 chains × 20k iters, beam width=10 depth=200 (spawn-parallel)")

if CHECKPOINT.exists():
    print("zone_c_decipherment already complete — skipping")
else:
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=STEP_TIMEOUT)
        if proc.returncode == 0:
            CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
        else:
            print(f"WARNING: zone_c_decipherment failed (exit {proc.returncode}) — continuing")
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"WARNING: zone_c_decipherment timed out after {STEP_TIMEOUT // 60} min — skipping")

# ── Display top hypotheses ─────────────────────────────────────────────────────
ranking_path = Path(f"{REPO_DIR}/outputs/decipherment/ranking.json")
if ranking_path.exists():
    ranking = json.loads(ranking_path.read_text())
    top = ranking.get("top_hypotheses", ranking if isinstance(ranking, list) else [])
    print(f"\n── Top {min(5, len(top))} hypotheses ──────────────────────────────────────")
    for i, h in enumerate(top[:5], 1):
        score = h.get("normalised_score", h.get("score", 0))
        n_signs = len(h.get("phoneme_map", {}))
        print(f"  #{i}  score={score:.4f}  signs={n_signs}")
        sample = dict(list(h.get("phoneme_map", {}).items())[:6])
        print(f"       {sample}")

# ── Save to Drive ─────────────────────────────────────────────────────────────
decipher_dir = Path(f"{REPO_DIR}/outputs/decipherment")
if decipher_dir.exists():
    decipher_out = Path(CHECKPOINTS_DIR) / "decipherment"
    decipher_out.mkdir(parents=True, exist_ok=True)
    for fname in ["ranking.json", "ranking.csv", "ranking.md", "decipherment_report.html"]:
        src = decipher_dir / fname
        if src.exists():
            shutil.copy(src, decipher_out / fname)
            print(f"  ✓ Saved {fname} to Drive")
    hyp_files = sorted(decipher_dir.glob("hypothesis_*.json"))
    for f in hyp_files[:10]:
        shutil.copy(f, decipher_out / f.name)
    if hyp_files:
        print(f"  ✓ {len(hyp_files)} hypothesis files saved to Drive")

print("\nFinal Cell 1 (save + download) is ready to run.")

In [ ]:
# ── Zone C Cell 2: Gumbel-Softmax gradient refinement of MCMC assignments ─────
# Refines the top-K MCMC hypotheses with ~100 gradient steps through a
# differentiable soft-LM scorer (add-α smoothed bigram transition matrix).
# Adds refined_assignments, refined_soft_score, and refined_lm_score to
# each hypothesis in ranking.json for honest comparison with the MCMC baseline.
#
# Requires Zone C Cell 1 to have completed (ranking.json must exist).
#
# N_STEPS = 10 → fast smoke-test validation (~5 sec)
# N_STEPS = 100 → full refinement (~30 sec per hypothesis on CPU)
# TOP_K = 5 → refine all 5 ranked hypotheses

import os, subprocess, sys, json, math
from pathlib import Path

STEP_TIMEOUT = 600  # seconds; kill and skip if exceeded
REPO_DIR         = "/content/repo"
CHECKPOINTS_DIR  = "/content/drive/MyDrive/hackingrongo_checkpoints"
DECIPHERMENT_OUT = f"{REPO_DIR}/outputs/decipherment"

N_STEPS = 100   # gradient steps per hypothesis
TOP_K   = 5     # number of top hypotheses to refine

os.chdir(REPO_DIR)

# ── Gate check: ranking.json present ─────────────────────────────────────────
ranking_path = Path(DECIPHERMENT_OUT) / "ranking.json"
if not ranking_path.exists():
    raise RuntimeError(
        "ranking.json not found — run Zone C Cell 1 first."
    )

lm_dir = Path(f"{REPO_DIR}/data/language_models")

cmd = [
    sys.executable, "scripts/refine_assignments.py",
    "--ranking",      str(ranking_path),
    "--lm-dir",       str(lm_dir),
    "--n-steps",      str(N_STEPS),
    "--top-k",        str(TOP_K),
    "--project-root", REPO_DIR,
    "--output",       str(ranking_path),   # overwrite in-place
]

print("=" * 60)
print(f"ZONE C GRADIENT REFINEMENT — {N_STEPS} steps × top-{TOP_K} hypotheses")
print("=" * 60)

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
try:
    proc.wait(timeout=STEP_TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    print(f"WARNING: Step timed out after {{STEP_TIMEOUT}}s — continuing")
    proc.returncode = -1
if proc.returncode != 0:
    raise RuntimeError(f"Gradient refinement failed (exit {proc.returncode})")

# ── Parse and display refinement results ─────────────────────────────────────
print("\n" + "=" * 60)
print("REFINEMENT RESULTS")
print("=" * 60)

ranking = json.loads(ranking_path.read_text())
hyps = ranking.get("hypotheses", [])[:TOP_K]

print(f"{'Rank':<6}{'Hypothesis':<12}{'Orig LM':>12}{'Refined LM':>12}{'Delta':>10}")
print("-" * 52)
for i, h in enumerate(hyps):
    orig  = h.get("overall_lm_score")
    ref   = h.get("refined_lm_score")
    delta = h.get("refinement_delta_lm")
    hid   = h.get("hypothesis_id", f"H{i:04d}")
    orig_s  = f"{orig:.4f}"  if orig  is not None else "N/A"
    ref_s   = f"{ref:.4f}"   if ref   is not None else "N/A"
    delta_s = f"{delta:+.4f}" if delta is not None else "N/A"
    print(f"{i+1:<6}{hid:<12}{orig_s:>12}{ref_s:>12}{delta_s:>10}")

print("\nRefinement complete. ranking.json updated with refined_assignments.")
import shutil
shutil.copy(ranking_path, f"{CHECKPOINTS_DIR}/ranking.json")
print(f"  ✓ ranking.json copied to checkpoints ({ranking_path.stat().st_size/1024:.0f} KB)")


In [ ]:
# ── Zone C Cell 3: Ensemble self-training loop ───────────────────────────────
# Iteratively expands the calendar anchor set by promoting consensus assignments
# back into MCMC as soft and hard cribs — then re-runs to convergence.
#
# Each round:
#   MCMC + beam → score top-K hypotheses → extract consensus assignments
#   → soft promotion (phoneme boost + IC damp) or hard graduation (crib)
#   → repeat with expanded anchor set
#
# SMOKE_TEST    = True  → 1 chain × 200 iter, 2 self-training rounds  (~5 min)
# MAX_ITERATIONS        → full run (default 5 rounds, ~5× MCMC wall time)
# MIN_CONSENSUS         → signs need ≥ this many of top-K to agree for soft promotion
# THRESHOLD_START/END   → confidence floor, relaxes linearly over iterations
#
# Requires: Zone C Cell 1 complete (ranking.json must exist).
#
# Outputs written to Drive:
#   self_training_summary.json
#   self_training_report.html
#   iter_NN/ranking.json  (per-iteration hypothesis rankings)

import os, subprocess, sys, json, shutil, math
from datetime import datetime, timezone
from pathlib import Path

STEP_TIMEOUT = 3600   # seconds; kill and skip if exceeded
REPO_DIR         = "/content/repo"
CHECKPOINTS_DIR  = "/content/drive/MyDrive/hackingrongo_checkpoints"
ST_OUT           = f"{REPO_DIR}/outputs/self_training"
CHECKPOINT       = Path(f"{REPO_DIR}/outputs/checkpoints/self_training.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

SMOKE_TEST       = False    # ← True: 1 chain × 200 iter, 2 rounds (~5 min)
MEDIUM_RUN       = True     # ← True: 2 chains × 5000 iter per round; False: 4 chains × 50k (hours)
MAX_ITERATIONS   = 5       # ignored when SMOKE_TEST = True (capped to 2)
TOP_K            = 5       # hypotheses used for consensus extraction
MIN_CONSENSUS    = 2       # min agreeing hypotheses for soft promotion
THRESHOLD_START  = 0.90    # confidence floor, iteration 0
THRESHOLD_END    = 0.70    # confidence floor, final iteration

os.chdir(REPO_DIR)

# ── Gate check ────────────────────────────────────────────────────────────────
ranking_path = Path(f"{REPO_DIR}/outputs/decipherment/ranking.json")
if not ranking_path.exists():
    raise RuntimeError(
        "ranking.json not found — run Zone C Cell 1 (MCMC) first."
    )

cmd = [
    sys.executable, "scripts/run_self_training.py",
    "--top-k",           str(TOP_K),
    "--min-consensus",   str(MIN_CONSENSUS),
    "--threshold-start", str(THRESHOLD_START),
    "--threshold-end",   str(THRESHOLD_END),
    "--output-dir",      ST_OUT,
]
if SMOKE_TEST:
    cmd.append("--smoke-test")
else:
    cmd += ["--max-iterations", str(MAX_ITERATIONS)]
    if MEDIUM_RUN:
        cmd += ["--chains", "2", "--iterations", "5000"]

mode_str = (
    "SMOKE TEST (1 chain × 200 iter, 2 rounds)" if SMOKE_TEST
    else f"MEDIUM ({MAX_ITERATIONS} rounds × 2 chains × 5k iter)" if MEDIUM_RUN
    else f"FULL RUN ({MAX_ITERATIONS} rounds × top-{TOP_K})"
)

print("=" * 60)
print(f"ZONE C SELF-TRAINING — {mode_str}")
print("=" * 60)

if CHECKPOINT.exists():
    print("self_training already complete — skipping")
else:
    proc = subprocess.Popen(
        cmd, cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=STEP_TIMEOUT)
        if proc.returncode == 0:
            CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
        else:
            print(f"WARNING: self_training failed (exit {proc.returncode}) — continuing")
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"WARNING: self_training timed out after {STEP_TIMEOUT // 60} min — skipping")

# ── Display summary ───────────────────────────────────────────────────────────
summary_path = Path(ST_OUT) / "self_training_summary.json"
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    print("\n" + "=" * 60)
    print("SELF-TRAINING SUMMARY")
    print("=" * 60)
    print(f"  Convergence   : {s['convergence']}")
    print(f"  Iterations    : {s['n_iterations']}")
    print(f"  Hard cribs    : {len(s['self_training_hard_cribs'])}")
    print(f"  Soft anchors  : {len(s['residual_soft_anchors'])}")
    traj = s.get("score_trajectory", [])
    if len(traj) >= 2:
        print(f"  Score delta   : {traj[-1] - traj[0]:+.6f}"
              f"  ({traj[0]:.4f} → {traj[-1]:.4f})")
    print("\n  Iteration breakdown:")
    for r in s["history"]:
        print(f"    iter {r['iteration']}: LM={r['top_lm_score']:.4f}"
              f"  +hard={len(r['new_hard'])}  +soft={len(r['new_soft'])}"
              f"  cribs={r['n_hard_cribs']}")
    if s["self_training_hard_cribs"]:
        print("\n  New hard cribs promoted:")
        for sign, ph in s["self_training_hard_cribs"].items():
            print(f"    {sign} → {ph}")
    if SMOKE_TEST:
        print("\nSmoke test passed. Set SMOKE_TEST = False for a full run.")
else:
    print("self_training_summary.json not found — check logs above")

# ── Copy outputs to checkpoints ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("Copying to Drive")
print("=" * 60)
for fname in ["self_training_summary.json", "self_training_report.html"]:
    src = Path(ST_OUT) / fname
    if src.exists():
        shutil.copy(src, f"{CHECKPOINTS_DIR}/{fname}")
        print(f"  ✓ {fname}  {src.stat().st_size/1024:.0f} KB")
    else:
        print(f"  — {fname}  (not generated)")

# Copy per-iteration rankings
iter_dirs = sorted(Path(ST_OUT).glob("iter_*/"))
for d in iter_dirs:
    rj = d / "ranking.json"
    if rj.exists():
        dest_name = f"st_{d.name}_ranking.json"
        shutil.copy(rj, f"{CHECKPOINTS_DIR}/{dest_name}")
        print(f"  ✓ {dest_name}")


In [ ]:
# ── RedRongo Cell: Autonomous red team agent ──────────────────────────────────
# Runs the RedRongo agent autonomously. The agent decides which attack
# strategies to deploy. Each turn is logged to MLflow and outputs/redteam/.
# Expected duration: 10-30 minutes depending on --max-turns.
# Requires: ANTHROPIC_API_KEY set as Colab secret (Secrets tab)

import subprocess, sys, os, json
from datetime import datetime, timezone
from pathlib import Path

REPO_DIR = "/content/repo"
STEP_TIMEOUT = 2400  # seconds; kill and skip if exceeded
CHECKPOINT   = Path(f"{REPO_DIR}/outputs/checkpoints/redteam_agent.done")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

# Read API key from Colab secrets
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = api_key

if CHECKPOINT.exists():
    print("redteam_agent already complete — skipping")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/redteam_agent.py",
            "--max-turns", "12",
            "--output", f"{REPO_DIR}/outputs/redteam/",
        ],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO_DIR
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=STEP_TIMEOUT)
        if proc.returncode == 0:
            CHECKPOINT.write_text(datetime.now(timezone.utc).isoformat())
        else:
            print(f"WARNING: redteam_agent failed (exit {proc.returncode}) — continuing")
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"WARNING: redteam_agent timed out after {STEP_TIMEOUT // 60} min — skipping")


In [ ]:
# ── Cell 8c: HSP group structure analysis ────────────────────────────────
# Extracts sign substitution pairs from parallel passage variants,
# tests group closure, finds a minimal generator set, and assesses
# feasibility of Hidden Subgroup Problem framing for quantum decipherment.
#
# Requires: Zone B Cell 2 (parallel_variants_auto.json generated).
# Output saved to Drive: hsp_analysis.json

import subprocess, sys, os, json, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
os.chdir(REPO_DIR)

VARIANTS_PATH = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
HSP_OUT       = Path(f"{REPO_DIR}/outputs/hsp_analysis.json")

if not VARIANTS_PATH.exists():
    print("parallel_variants_auto.json not found — run Zone B Cell 2 first.")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/hsp_analysis.py",
            "--parallels", str(VARIANTS_PATH),
            "--output",    str(HSP_OUT),
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: hsp_analysis failed (exit {proc.returncode})")

    if HSP_OUT.exists():
        result = json.loads(HSP_OUT.read_text())
        feas   = result.get("hsp_feasibility", {})
        subs   = result.get("substitutions", {})
        gens   = result.get("generators", [])
        clos   = result.get("closure", {})
        print(f"\n── HSP feasibility: {feas.get('level', '?').upper()} ──────────")
        print(f"  {feas.get('rationale', '')}")
        print(f"  Observed substitutions : {subs.get('n_pairs', '?')}")
        print(f"  Closure ratio          : {clos.get('closure_ratio', 0):.2f}")
        print(f"  Generator set size     : {len(gens)}")
        if gens:
            print(f"  Generators: {gens[:8]}")
        shutil.copy(HSP_OUT, Path(CHECKPOINTS_DIR) / "hsp_analysis.json")
        print(f"  ✓ hsp_analysis.json saved to Drive")

In [ ]:
# ── Zone C Cell 4: Tablet D reconstruction ──────────────────────────────────
# Runs the Échancrée (Tablet D) uncertain-sign reconstruction pipeline.
# Requires: Zone C Cell 2 (ranking.json) and Zone A Cell 2 (sequence_model.json).
# Gracefully degrades if either is missing — always writes JSON + HTML.
#
# Outputs saved to Drive:
#   reconstruction/tablet_d_reconstruction.json
#   reconstruction/tablet_d_reconstruction_report.html

import subprocess, sys, os, shutil, json
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
os.chdir(REPO_DIR)

proc = subprocess.Popen(
    [sys.executable, "scripts/reconstruct_tablet_d.py",
     "--corpus-dir", f"{REPO_DIR}/data/corpus",
     "--model",      f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
     "--ranking",    f"{REPO_DIR}/outputs/decipherment/ranking.json",
     "--glyphs-dir", f"{REPO_DIR}/data/glyphs",
     "--output",     f"{REPO_DIR}/outputs/reconstruction/"],
    cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

result_path = Path(f"{REPO_DIR}/outputs/reconstruction/tablet_d_reconstruction.json")
if result_path.exists():
    result = json.loads(result_path.read_text())
    summary = result.get("summary", {})
    print(f"\n── Tablet D Reconstruction Summary ─────────────────────────")
    print(f"  Targets identified    : {summary.get('n_targets', 0)}")
    print(f"  Convergent candidates : {summary.get('n_convergent', 0)}")
    print(f"  Top candidate         : {summary.get('top_candidate', '—')}")
    candidates = result.get("convergent_candidates", [])
    for c in candidates:
        print(f"  {c['barthel_raw']:<10} score={c['convergence_score']:.2f}  "
              f"mcmc={c.get('mcmc_phoneme','—')}  "
              f"seq_top1={c['sequence_top_k'][0]['sign'] if c['sequence_top_k'] else '—'}")

recon_ckpt = Path(CHECKPOINTS_DIR) / "reconstruction"
recon_ckpt.mkdir(exist_ok=True)
for fname in ["tablet_d_reconstruction.json", "tablet_d_reconstruction_report.html"]:
    src = Path(f"{REPO_DIR}/outputs/reconstruction/{fname}")
    if src.exists():
        shutil.copy(src, recon_ckpt / fname)
        print(f"  \u2713 Saved {fname} to Drive")

In [ ]:
# ── Zone C Cell X: Matched-size subsample test for H0001 ─────────────────────
# Bootstrap test: compare pre-contact LM score vs post-contact distribution
# at matched sample size (n=88), repeated 1,000 times.

import os, sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

REPO_DIR = "/content/repo"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from hackingrongo.data.corpus import load_corpus, split_by_cluster
from hackingrongo.zone_c.lm_scoring import LMScorer

# Parameters
HYPOTHESIS_ID = "H0001"
SAMPLE_SIZE = 88
N_BOOT = 1000
SEED = 42

cfg = OmegaConf.load(f"{REPO_DIR}/conf/config.yaml")
project_root = Path(REPO_DIR)

# Load hypothesis map from ranking
ranking_path = project_root / "outputs" / "decipherment" / "ranking.json"
if not ranking_path.exists():
    raise FileNotFoundError(f"Missing ranking.json: {ranking_path}")

ranking = json.loads(ranking_path.read_text(encoding="utf-8"))
hypotheses = ranking.get("hypotheses", [])
if not hypotheses:
    raise RuntimeError("ranking.json has no hypotheses")

hyp = next((h for h in hypotheses if h.get("hypothesis_id") == HYPOTHESIS_ID), None)
if hyp is None:
    raise RuntimeError(f"Hypothesis {HYPOTHESIS_ID} not found in ranking.json")

phoneme_map = {
    a["sign_code"]: a["phoneme"]
    for a in hyp.get("assignments", [])
    if isinstance(a, dict) and "sign_code" in a and "phoneme" in a
}
if not phoneme_map:
    raise RuntimeError(f"No assignments found for {HYPOTHESIS_ID}")

# Load corpus and split by stratum
all_tablets = load_corpus(cfg, project_root)
by = split_by_cluster(all_tablets)

pre_tokens = [tok.barthel_code for t in by.get("pre_contact", []) for tok in t.tokens]
post_tokens = [tok.barthel_code for t in by.get("post_contact", []) for tok in t.tokens]

if len(post_tokens) < SAMPLE_SIZE:
    raise RuntimeError(
        f"post_contact has {len(post_tokens)} tokens < SAMPLE_SIZE={SAMPLE_SIZE}"
    )
if not pre_tokens:
    raise RuntimeError("No pre_contact tokens found")

lm_scorer = LMScorer(cfg, project_root)

def lm_score_for_tokens(glyph_tokens):
    phoneme_seq = [phoneme_map.get(g, "<UNK>") for g in glyph_tokens]
    return float(lm_scorer.score(phoneme_seq).ensemble_log_prob)

# Pre-contact reference score
pre_score_full = lm_score_for_tokens(pre_tokens)

rng = np.random.default_rng(SEED)
if len(pre_tokens) >= SAMPLE_SIZE:
    pre_tokens_matched = rng.choice(pre_tokens, size=SAMPLE_SIZE, replace=False).tolist()
    pre_score_matched = lm_score_for_tokens(pre_tokens_matched)
    pre_ref_score = pre_score_matched
    pre_ref_label = f"pre-contact sampled n={SAMPLE_SIZE}"
else:
    pre_score_matched = None
    pre_ref_score = pre_score_full
    pre_ref_label = f"pre-contact full n={len(pre_tokens)}"

# Bootstrap post-contact matched-size distribution
post_scores = np.empty(N_BOOT, dtype=float)
for i in range(N_BOOT):
    sample = rng.choice(post_tokens, size=SAMPLE_SIZE, replace=False).tolist()
    post_scores[i] = lm_score_for_tokens(sample)

# One-sided empirical p-value: how often post-contact >= pre-contact reference
p_ge = float(np.mean(post_scores >= pre_ref_score))
percentile = float(np.mean(post_scores < pre_ref_score) * 100.0)
q = np.quantile(post_scores, [0.025, 0.05, 0.5, 0.95, 0.975])

print("=" * 72)
print(f"Matched-size subsample test under {HYPOTHESIS_ID}")
print("=" * 72)
print(f"pre tokens:  {len(pre_tokens)}")
print(f"post tokens: {len(post_tokens)}")
print(f"sample size per draw: {SAMPLE_SIZE}")
print(f"bootstrap draws:      {N_BOOT}")
print()
print(f"pre_score_full:    {pre_score_full:.6f}")
if pre_score_matched is not None:
    print(f"pre_score_matched: {pre_score_matched:.6f}")
print(f"reference used:    {pre_ref_label} = {pre_ref_score:.6f}")
print()
print("post-contact bootstrap distribution:")
print(f"  mean={post_scores.mean():.6f}  std={post_scores.std(ddof=1):.6f}")
print(f"  q2.5={q[0]:.6f}  q5={q[1]:.6f}  q50={q[2]:.6f}  q95={q[3]:.6f}  q97.5={q[4]:.6f}")
print()
print(f"P(post >= pre_ref) = {p_ge:.4f}")
print(f"pre_ref percentile in post distribution = {percentile:.2f}%")

if pre_ref_score > q[3]:
    print("Conclusion: pre-contact score is above the post-contact 95th percentile (likely real signal).")
elif pre_ref_score < q[1]:
    print("Conclusion: pre-contact score is below the post-contact 5th percentile.")
else:
    print("Conclusion: pre-contact score lies within the post-contact matched-size distribution (sample-size effect plausible).")

# Plot
plt.figure(figsize=(8, 4.5))
plt.hist(post_scores, bins=40, alpha=0.75, color="#4C72B0", edgecolor="white")
plt.axvline(pre_ref_score, color="#C44E52", linestyle="--", linewidth=2,
            label=f"pre reference ({pre_ref_label})")
plt.axvline(post_scores.mean(), color="#55A868", linestyle="-", linewidth=2,
            label="post bootstrap mean")
plt.title(f"Post-contact LM scores at n={SAMPLE_SIZE} (N={N_BOOT}), {HYPOTHESIS_ID}")
plt.xlabel("Ensemble LM score")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

# Optional save for report/traceability
out = {
    "hypothesis_id": HYPOTHESIS_ID,
    "sample_size": SAMPLE_SIZE,
    "n_bootstrap": N_BOOT,
    "seed": SEED,
    "n_pre_tokens": len(pre_tokens),
    "n_post_tokens": len(post_tokens),
    "pre_score_full": pre_score_full,
    "pre_score_matched": pre_score_matched,
    "pre_reference_label": pre_ref_label,
    "pre_reference_score": pre_ref_score,
    "post_mean": float(post_scores.mean()),
    "post_std": float(post_scores.std(ddof=1)),
    "post_quantiles": {
        "q2_5": float(q[0]), "q5": float(q[1]), "q50": float(q[2]),
        "q95": float(q[3]), "q97_5": float(q[4]),
    },
    "p_post_ge_pre": p_ge,
    "pre_percentile_in_post": percentile,
}

out_path = project_root / "outputs" / "analysis" / "h0001_subsample_test.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, indent=2), encoding="utf-8")
print(f"\nSaved: {out_path}")

In [ ]:
# ── Final Cell: Bundle reports + save to Drive ───────────────────────────────
# Bundles every HTML report into reports_bundle.zip with a navigable index.
# Saves to Drive and downloads the bundle in ONE click.
#
# Download workflow:
#   1) Run this cell — wait for "bundle complete"
#   2) reports_bundle.zip downloads automatically to your browser Downloads
#   3) Unzip → open index.html in any browser

import os, subprocess, sys, json, shutil
from pathlib import Path
from google.colab import files

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
DRIVE_DIR       = Path(CHECKPOINTS_DIR)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)

# ── Step 1: Bundle all HTML reports ──────────────────────────────────────────
print("=" * 60)
print("STEP 1: Bundling HTML reports")
print("=" * 60)

proc = subprocess.run(
    [sys.executable, "scripts/bundle_reports.py",
     "--outputs-dir", f"{REPO_DIR}/outputs"],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-500:])

bundle_zip = Path(f"{REPO_DIR}/outputs/reports_bundle.zip")
if bundle_zip.exists():
    shutil.copy(bundle_zip, DRIVE_DIR / "reports_bundle.zip")
    print(f"  ✓ reports_bundle.zip saved to Drive  ({bundle_zip.stat().st_size/1024:.0f} KB)")
else:
    print("  — reports_bundle.zip not generated")

# ── Step 2: Save non-HTML outputs to Drive ────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Saving other outputs to Drive")
print("=" * 60)

def _save(src: Path, label: str) -> bool:
    if src.exists():
        shutil.copy(src, DRIVE_DIR / src.name)
        print(f"  ✓ {label:45s} {src.stat().st_size/1024:.0f} KB")
        return True
    print(f"  — {label:45s} not generated")
    return False

# Checkpoints
for f in sorted(Path(f"{REPO_DIR}/outputs/checkpoints").glob("*.pt")):
    _save(f, f.name)

# Zone A
for fname in ["umap_embeddings.png", "cluster_vs_barthel.csv", "cluster_vs_barthel.json"]:
    _save(Path(f"{REPO_DIR}/outputs/analysis/{fname}"), fname)

# Zone B
for fname in ["compound_candidates.json", "compound_validation.json", "spectrum_scores.json"]:
    _save(Path(f"{REPO_DIR}/outputs/analysis/{fname}"), fname)
_save(Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json"), "parallel_variants_auto.json")
_save(Path(f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json"), "astronomical_candidates.json")
_save(Path(f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json"), "pgood_analysis.json")

# Zone C
decipher_dir = Path(f"{REPO_DIR}/outputs/decipherment")
decipher_out = DRIVE_DIR / "decipherment"
decipher_out.mkdir(exist_ok=True)
for fname in ["ranking.json", "ranking.csv", "ranking.md"]:
    src = decipher_dir / fname
    if src.exists():
        shutil.copy(src, decipher_out / fname)
        print(f"  ✓ {fname:45s} {src.stat().st_size/1024:.0f} KB")
    else:
        print(f"  — {fname:45s} not generated")
if decipher_dir.exists():
    for f in sorted(decipher_dir.glob("hypothesis_*.json"))[:10]:
        shutil.copy(f, decipher_out / f.name)

# Language models
lm_dir = Path(f"{DATA_DIR}/language_models")
if lm_dir.exists():
    lm_out = DRIVE_DIR / "language_models"
    lm_out.mkdir(exist_ok=True)
    for f in sorted(lm_dir.glob("*.json")):
        shutil.copy(f, lm_out / f.name)
    print(f"  ✓ {'language_models/':45s} {len(list(lm_out.iterdir()))} files")

# Sensitivity + misc
for fname in ["sensitivity_analysis.json", "contact_partition.json"]:
    _save(Path(f"{REPO_DIR}/outputs/{fname}"), fname)
for fname in ["sequence_model.json", "tablet_f_reconstruction.json"]:
    _save(Path(f"{REPO_DIR}/outputs/zone_b/{fname}"), fname)

# Self-training
st_dir = Path(f"{REPO_DIR}/outputs/self_training")
_save(st_dir / "self_training_summary.json", "self_training_summary.json")
if st_dir.exists():
    for d in sorted(st_dir.glob("iter_*/")):
        rj = d / "ranking.json"
        if rj.exists():
            shutil.copy(rj, DRIVE_DIR / f"st_{d.name}_ranking.json")

# ── Step 3: Download reports_bundle.zip ───────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Downloading reports_bundle.zip to local machine")
print("=" * 60)

if bundle_zip.exists():
    print(f"  Triggering download of reports_bundle.zip ({bundle_zip.stat().st_size/1024:.0f} KB)…")
    files.download(str(bundle_zip))
    print("  ⬇ Download triggered — check your browser Downloads folder")
else:
    print("  — bundle not available, downloading individual reports as fallback")
    for f in sorted(Path(f"{REPO_DIR}/outputs").rglob("*.html")):
        files.download(str(f))

print()
print("  ┌── Review instructions ──────────────────────────────────────┐")
print("  │  1) Unzip reports_bundle.zip                                │")
print("  │  2) Open  index.html  in your browser                       │")
print("  │  3) All 30+ reports are linked — click to open              │")
print("  │  Bundle also saved to:                                       │")
print(f"  │    {CHECKPOINTS_DIR}/reports_bundle.zip")
print("  └─────────────────────────────────────────────────────────────┘")


In [ ]:
# ── MLflow UI — view experiment dashboard before session ends ────────────────
# Run this cell any time after training to inspect runs, compare metrics,
# and see the lm_score time-series for the self-training loop.
#
# The URL printed below opens the MLflow UI through Colab's port proxy.
# It stays active until the Colab session ends.

import subprocess, time, os
from pathlib import Path

REPO_DIR = "/content/repo"

# Start MLflow UI server in the background
_mlflow_proc = subprocess.Popen(
    ["mlflow", "ui",
     "--host", "0.0.0.0",
     "--port", "5000",
     "--backend-store-uri", os.environ.get("MLFLOW_TRACKING_URI", "outputs/mlruns")],
    cwd=REPO_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # wait for server to bind

# Get Colab proxy URL
try:
    from google.colab.output import eval_js
    proxy_url = eval_js("google.colab.kernel.proxyPort(5000)")
    print("=" * 60)
    print("  MLflow UI is live:")
    print(f"  {proxy_url}")
    print()
    print("  Experiments: rongorongo_decipherment")
    print("  Click any run to see metrics + artifacts")
    print("  Self-training runs show lm_score as a time-series chart")
    print("=" * 60)
except Exception as e:
    print(f"Port proxy unavailable ({e}). Access MLflow at http://localhost:5000")
    print("(Works if you are running this locally.)")


In [ ]:
# ── Pozdniakov Hypothesis Tests ─────────────────────────────────────────────
# Tests the syllabic-only hypothesis against a mixed syllabic+logographic view.
# Uses H0001 from outputs/decipherment/ranking.json.

import json
import math
import os
import statistics
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf

REPO_DIR = "/content/repo"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from hackingrongo.data.corpus import load_corpus, split_by_cluster
from hackingrongo.data.rapa_nui_corpus import NGramLM
from hackingrongo.results.schema import load_ranking
from hackingrongo.zone_c.lm_scoring import LMScorer

PROJECT_ROOT = Path(REPO_DIR)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = OmegaConf.load(PROJECT_ROOT / "conf" / "config.yaml")
RNG = np.random.default_rng(42)
SAMPLE_SIZE = 88
N_BOOT = 1000
N_NULL = 1000
HYPOTHESIS_ID = "H0001"

LM_SCORER = LMScorer(CFG, PROJECT_ROOT)

ranking = load_ranking(PROJECT_ROOT / "outputs" / "decipherment" / "ranking.json")
hyp = next((h for h in ranking.hypotheses if h.hypothesis_id == HYPOTHESIS_ID), ranking.hypotheses[0])
phoneme_map = {a.sign_code: a.phoneme for a in hyp.assignments}

all_tablets = load_corpus(CFG, PROJECT_ROOT)
by_stratum = split_by_cluster(all_tablets)

# Load LM reference frequencies from the post-contact LM.
post_lm_path = PROJECT_ROOT / "data" / "language_models" / "post_contact_lm.json"
if not post_lm_path.exists():
    raise FileNotFoundError(f"Missing LM file: {post_lm_path}")
post_lm = NGramLM.load(post_lm_path)
ref_counts = dict(post_lm._counts[1].get((), {}))
ref_counts = {k: int(v) for k, v in ref_counts.items() if not k.startswith("<") and v > 0}
if not ref_counts:
    raise RuntimeError("Could not extract unigram frequencies from post_contact LM")
ref_phoneme_order = [ph for ph, _ in sorted(ref_counts.items(), key=lambda kv: (-kv[1], kv[0]))]
ref_phoneme_freq = {ph: ref_counts[ph] for ph in ref_counts}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _rankdata(values: list[float]) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    order = np.argsort(arr, kind="mergesort")
    ranks = np.empty(len(arr), dtype=float)
    i = 0
    while i < len(arr):
        j = i
        while j + 1 < len(arr) and arr[order[j + 1]] == arr[order[i]]:
            j += 1
        ranks[order[i : j + 1]] = (i + j + 2) / 2.0
        i = j + 1
    return ranks


def spearman_corr(x: list[float], y: list[float]) -> float:
    if len(x) < 2 or len(y) < 2 or len(x) != len(y):
        return float("nan")
    rx = _rankdata(x)
    ry = _rankdata(y)
    if np.std(rx) == 0 or np.std(ry) == 0:
        return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])


def bootstrap_ci(values: list[float], alpha: float = 0.05) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (float("nan"), float("nan"))
    return (float(np.quantile(arr, alpha / 2.0)), float(np.quantile(arr, 1.0 - alpha / 2.0)))


def levenshtein_distance(a: list[str], b: list[str]) -> int:
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost))
        prev = curr
    return prev[-1]


def norm_levenshtein(a: list[str], b: list[str]) -> float:
    denom = max(len(a), len(b), 1)
    return levenshtein_distance(a, b) / denom


def fit_zipf_alpha(counts: list[int]) -> float:
    freqs = np.asarray(sorted(counts, reverse=True), dtype=float)
    freqs = freqs[freqs > 0]
    ranks = np.arange(1, len(freqs) + 1, dtype=float)
    if len(freqs) < 2:
        return float("nan")
    slope, _ = np.polyfit(np.log(ranks), np.log(freqs), 1)
    return float(-slope)


def make_sign_freqs(tokens: list[str]) -> Counter:
    return Counter(tokens)


def sign_to_phoneme_rho(tokens: list[str]) -> float:
    sign_counts = make_sign_freqs(tokens)
    if not sign_counts:
        return float("nan")
    sign_order = sorted(sign_counts, key=lambda s: (-sign_counts[s], s))
    sign_rank = {s: i + 1 for i, s in enumerate(sign_order)}
    x: list[float] = []
    y: list[float] = []
    for sign in sign_order:
        ph = phoneme_map.get(sign)
        if ph is None or ph not in ref_phoneme_freq:
            continue
        x.append(sign_rank[sign])
        y.append(ref_phoneme_freq[ph])
    return spearman_corr(x, y) if len(x) >= 5 else float("nan")


def lm_score_tablet(tablet) -> float:
    seq = [phoneme_map.get(tok.barthel_code, "<UNK>") for tok in tablet.tokens]
    return float(LM_SCORER.score(seq).ensemble_log_prob)


# ---------------------------------------------------------------------------
# Test 1 — pre-contact frequency correlation vs post-contact bootstrap
# ---------------------------------------------------------------------------

pre_tokens = [tok.barthel_code for t in by_stratum.get("pre_contact", []) for tok in t.tokens]
post_tokens = [tok.barthel_code for t in by_stratum.get("post_contact", []) for tok in t.tokens]
unknown_tokens = [tok.barthel_code for t in by_stratum.get("unknown", []) for tok in t.tokens]

if len(pre_tokens) < SAMPLE_SIZE:
    raise RuntimeError(f"Need at least {SAMPLE_SIZE} pre-contact tokens, found {len(pre_tokens)}")
if len(post_tokens) < SAMPLE_SIZE:
    raise RuntimeError(f"Need at least {SAMPLE_SIZE} post-contact tokens, found {len(post_tokens)}")

pre_sample = pre_tokens[:SAMPLE_SIZE]
pre_rho = sign_to_phoneme_rho(pre_sample)
pre_boot = []
for _ in range(N_BOOT):
    sample = RNG.choice(pre_tokens, size=SAMPLE_SIZE, replace=True).tolist()
    pre_boot.append(sign_to_phoneme_rho(sample))
pre_boot = [v for v in pre_boot if np.isfinite(v)]
pre_ci = bootstrap_ci(pre_boot)

post_boot = []
for _ in range(N_BOOT):
    sample = RNG.choice(post_tokens, size=SAMPLE_SIZE, replace=True).tolist()
    post_boot.append(sign_to_phoneme_rho(sample))
post_boot = [v for v in post_boot if np.isfinite(v)]
post_ci = bootstrap_ci(post_boot)
post_full_rho = sign_to_phoneme_rho(post_tokens)

pre_vs_post_p = float(np.mean(np.asarray(post_boot) >= pre_rho)) if pre_boot else float("nan")
post_vs_pre_p = float(np.mean(np.asarray(pre_boot) >= post_full_rho)) if pre_boot else float("nan")

print("=" * 76)
print(f"Pozdniakov hypothesis tests under {HYPOTHESIS_ID}")
print("=" * 76)
print("Test 1 — frequency correlation (matched n=88)")
print(f"  pre-contact rho (n={len(pre_sample)}): {pre_rho:.4f}  95% CI {pre_ci[0]:.4f} to {pre_ci[1]:.4f}")
print(f"  post-contact rho bootstrap mean: {float(np.mean(post_boot)):.4f}  95% CI {post_ci[0]:.4f} to {post_ci[1]:.4f}")
print(f"  post-contact full rho: {post_full_rho:.4f}")
print(f"  P(post bootstrap >= pre rho) = {pre_vs_post_p:.4f}")
print(f"  P(pre bootstrap >= post full rho) = {post_vs_pre_p:.4f}")

# ---------------------------------------------------------------------------
# Test 2 — hapax rates by stratum
# ---------------------------------------------------------------------------

def hapax_rate(tokens: list[str]) -> dict[str, float]:
    counts = Counter(tokens)
    n_types = len(counts)
    n_hapax = sum(1 for c in counts.values() if c == 1)
    return {
        "n_tokens": len(tokens),
        "n_types": n_types,
        "n_hapax": n_hapax,
        "hapax_rate": n_hapax / n_types if n_types else float("nan"),
    }

hapax = {
    "pre_contact": hapax_rate(pre_tokens),
    "post_contact": hapax_rate(post_tokens),
    "unknown": hapax_rate(unknown_tokens),
}
print()
print("Test 2 — hapax legomena rate by stratum")
for label, vals in hapax.items():
    print(
        f"  {label:12s}: hapax {vals['n_hapax']:4d} / types {vals['n_types']:4d} "
        f"= {vals['hapax_rate']:.3f}"
    )

# ---------------------------------------------------------------------------
# Test 3 — passage stability analysis
# ---------------------------------------------------------------------------

parallel_candidates = [
    PROJECT_ROOT / "data" / "parallels" / "parallel_variants_auto.json",
    PROJECT_ROOT / "data" / "parallels" / "parallel_variants.json",
]
parallel_path = next((p for p in parallel_candidates if p.exists()), None)
if parallel_path is None:
    raise FileNotFoundError("No parallel passage JSON found")
parallel_data = json.loads(parallel_path.read_text(encoding="utf-8"))
passages = parallel_data.get("passages", parallel_data if isinstance(parallel_data, list) else [])

pre_post_dists: list[float] = []
post_post_dists: list[float] = []
passage_stability_rows: list[dict] = []

for entry in passages:
    if not isinstance(entry, dict):
        continue
    attestations = entry.get("attestations", entry.get("variants", []))
    by = defaultdict(list)
    for att in attestations:
        if not isinstance(att, dict):
            continue
        form = att.get("form", att.get("glyphs", []))
        if not form:
            continue
        seq = [str(g) for g in form]
        stratum = att.get("stratum", "unknown")
        by[stratum].append(seq)

    pre_forms = by.get("pre_contact", [])
    post_forms = by.get("post_contact", [])
    if pre_forms and post_forms:
        pp = [norm_levenshtein(a, b) for a in pre_forms for b in post_forms]
        qq = [norm_levenshtein(a, b) for i, a in enumerate(post_forms) for b in post_forms[i + 1 :]]
        if pp:
            pre_post_dists.extend(pp)
        if qq:
            post_post_dists.extend(qq)
        passage_stability_rows.append({
            "passage_id": entry.get("passage_id", entry.get("id", "?")),
            "n_pre": len(pre_forms),
            "n_post": len(post_forms),
            "pre_post_mean": float(np.mean(pp)) if pp else float("nan"),
            "post_post_mean": float(np.mean(qq)) if qq else float("nan"),
        })

print()
print("Test 3 — passage stability")
print(f"  passages with both pre/post attestations: {len(passage_stability_rows)}")
print(f"  mean pre-post normalized edit distance: {float(np.mean(pre_post_dists)):.4f}" if pre_post_dists else "  mean pre-post normalized edit distance: n/a")
print(f"  mean post-post normalized edit distance: {float(np.mean(post_post_dists)):.4f}" if post_post_dists else "  mean post-post normalized edit distance: n/a")

# ---------------------------------------------------------------------------
# Test 4 — tablet-level LM score gradient under H0001
# ---------------------------------------------------------------------------

tablet_rows = []
for tablet in all_tablets:
    if tablet.stratum == "excluded":
        continue
    score = lm_score_tablet(tablet)
    tablet_rows.append({
        "tablet_id": tablet.tablet_id,
        "stratum": tablet.stratum,
        "date_midpoint": float(tablet.date_midpoint),
        "n_tokens": len(tablet.tokens),
        "lm_score": score,
    })

tablet_rows.sort(key=lambda r: r["date_midpoint"])
dates = [r["date_midpoint"] for r in tablet_rows]
score_series = [r["lm_score"] for r in tablet_rows]
score_by_date_rho = spearman_corr(dates, score_series)
print()
print("Test 4 — tablet-level LM score gradient")
print(f"  Spearman rho(date, LM score): {score_by_date_rho:.4f}")
print(f"  best tablet: {max(tablet_rows, key=lambda r: r['lm_score'])['tablet_id']} @ {max(tablet_rows, key=lambda r: r['lm_score'])['lm_score']:.4f}")
print(f"  worst tablet: {min(tablet_rows, key=lambda r: r['lm_score'])['tablet_id']} @ {min(tablet_rows, key=lambda r: r['lm_score'])['lm_score']:.4f}")

# ---------------------------------------------------------------------------
# Test 5 — Zipf null model vs observed post-contact correlation
# ---------------------------------------------------------------------------

post_counts = Counter(post_tokens)
alpha_zipf = fit_zipf_alpha(list(post_counts.values()))
post_n_types = len(post_counts)
post_n_tokens = len(post_tokens)

if not np.isfinite(alpha_zipf):
    raise RuntimeError("Could not fit Zipf alpha for post-contact corpus")

ranks = np.arange(1, post_n_types + 1, dtype=float)
zipf_probs = ranks ** (-alpha_zipf)
zipf_probs = zipf_probs / zipf_probs.sum()

ref_phoneme_freqs = np.array([ref_counts[ph] for ph in ref_phoneme_order], dtype=float)
ref_phoneme_freqs = ref_phoneme_freqs / ref_phoneme_freqs.sum()

null_rhos = []
for _ in range(N_NULL):
    sampled_counts = RNG.multinomial(post_n_tokens, zipf_probs)
    sampled_counts = sampled_counts[sampled_counts > 0]
    sign_freqs = sampled_counts.tolist()

    perm = RNG.permutation(len(ref_phoneme_order))
    if len(sign_freqs) > len(ref_phoneme_order):
        ph_freqs = np.resize(ref_phoneme_freqs[perm], len(sign_freqs))
    else:
        ph_freqs = ref_phoneme_freqs[perm[: len(sign_freqs)]]
    rho = spearman_corr(sign_freqs, ph_freqs.tolist())
    if np.isfinite(rho):
        null_rhos.append(rho)

observed_post_rho = post_full_rho
null_mean = float(np.mean(null_rhos)) if null_rhos else float("nan")
null_ci = bootstrap_ci(null_rhos)
p_null_ge_obs = float(np.mean(np.asarray(null_rhos) >= observed_post_rho)) if null_rhos else float("nan")

print()
print("Test 5 — Zipf null model")
print(f"  fitted Zipf alpha (post-contact): {alpha_zipf:.4f}")
print(f"  observed post-contact rho: {observed_post_rho:.4f}")
print(f"  null mean rho: {null_mean:.4f}  95% CI {null_ci[0]:.4f} to {null_ci[1]:.4f}")
print(f"  P(null rho >= observed) = {p_null_ge_obs:.4f}")

# ---------------------------------------------------------------------------
# Plots
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes[0, 0]
ax.hist(post_boot, bins=40, alpha=0.75, color="#4C72B0", label="post bootstrap")
ax.axvline(pre_rho, color="#C44E52", linestyle="--", linewidth=2, label="pre rho")
ax.axvline(np.mean(post_boot), color="#55A868", linewidth=2, label="post mean")
ax.set_title("Test 1: matched-size frequency correlation")
ax.set_xlabel("Spearman rho")
ax.set_ylabel("Count")
ax.legend(fontsize=8)

ax = axes[0, 1]
labels = list(hapax)
rates = [hapax[k]["hapax_rate"] for k in labels]
ax.bar(labels, rates, color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_ylim(0, 1)
ax.set_title("Test 2: hapax rate by stratum")
ax.tick_params(axis="x", rotation=20)

ax = axes[1, 0]
if pre_post_dists and post_post_dists:
    ax.hist(pre_post_dists, bins=20, alpha=0.7, label="pre-post", color="#C44E52")
    ax.hist(post_post_dists, bins=20, alpha=0.7, label="post-post", color="#4C72B0")
ax.set_title("Test 3: passage stability")
ax.set_xlabel("Normalized edit distance")
ax.set_ylabel("Count")
ax.legend(fontsize=8)

ax = axes[1, 1]
if null_rhos:
    ax.hist(null_rhos, bins=40, alpha=0.75, color="#7B6FE8", label="Zipf null")
    ax.axvline(observed_post_rho, color="#C44E52", linewidth=2, label="observed post-contact")
ax.set_title("Test 5: Zipf null model")
ax.set_xlabel("Spearman rho")
ax.set_ylabel("Count")
ax.legend(fontsize=8)

fig.tight_layout()
fig_path = OUTPUT_DIR / "pozdniakov_hypothesis_tests.png"
fig.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 6))
colors = ["#4C72B0" if r["stratum"] == "pre_contact" else "#55A868" if r["stratum"] == "post_contact" else "#888888" for r in tablet_rows]
ax2.scatter(dates, score_series, c=colors, s=60, alpha=0.85, edgecolor="white", linewidth=0.5)
ax2.set_title("Test 4: LM score by tablet under H0001")
ax2.set_xlabel("Radiocarbon midpoint (CE)")
ax2.set_ylabel("H0001 LM score")
ax2.grid(alpha=0.2)
fig2.tight_layout()
score_fig_path = OUTPUT_DIR / "pozdniakov_tablet_scores.png"
fig2.savefig(score_fig_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# Persist results
# ---------------------------------------------------------------------------

results = {
    "hypothesis_id": HYPOTHESIS_ID,
    "sample_size": SAMPLE_SIZE,
    "n_bootstrap": N_BOOT,
    "n_null": N_NULL,
    "test1": {
        "pre_rho": pre_rho,
        "pre_ci": pre_ci,
        "post_boot_mean": float(np.mean(post_boot)) if post_boot else float("nan"),
        "post_ci": post_ci,
        "post_full_rho": post_full_rho,
        "p_post_boot_ge_pre": pre_vs_post_p,
        "p_pre_boot_ge_post_full": post_vs_pre_p,
    },
    "test2": hapax,
    "test3": {
        "n_passages_with_pre_and_post": len(passage_stability_rows),
        "mean_pre_post_edit_distance": float(np.mean(pre_post_dists)) if pre_post_dists else None,
        "mean_post_post_edit_distance": float(np.mean(post_post_dists)) if post_post_dists else None,
        "rows": passage_stability_rows,
    },
    "test4": {
        "spearman_rho_date_score": score_by_date_rho,
        "tablets": tablet_rows,
    },
    "test5": {
        "alpha_zipf_post_contact": alpha_zipf,
        "observed_post_rho": observed_post_rho,
        "null_mean_rho": null_mean,
        "null_ci": null_ci,
        "p_null_ge_observed": p_null_ge_obs,
    },
    "artifacts": {
        "summary_plot": str(fig_path),
        "tablet_score_plot": str(score_fig_path),
        "parallel_path": str(parallel_path),
    },
}

results_path = OUTPUT_DIR / "pozdniakov_hypothesis_tests.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print()
print(f"Saved results to {results_path}")
print(f"Saved plots to {fig_path} and {score_fig_path}")

## Pipeline Enhancements

Cells A–G: Mamari calendar alignment → anchored decipherment → gloss → reading-order v2 → inpainting → final report.

In [ ]:
# ── Cell A: Mamari calendar alignment ───────────────────────────────────────
import subprocess, json
from pathlib import Path

result = subprocess.run(
    ["python", "scripts/align_mamari_calendar.py"],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

ALIGNMENT_PATH = Path(PROJECT_ROOT) / "outputs/analysis/mamari_calendar_alignment.json"
alignment = json.loads(ALIGNMENT_PATH.read_text()) if ALIGNMENT_PATH.exists() else {}
print(f"Aligned {alignment.get('n_night_names', 0)} nights, "
      f"{alignment.get('summary', {}).get('n_anchor_matched', 0)} anchor matches, "
      f"{alignment.get('summary', {}).get('n_ambiguous', 0)} ambiguous")


In [ ]:
# ── Cell B: Update CALENDAR_ANCHORS from alignment output ───────────────────
# Reads the high-confidence anchors (confidence >= 0.87) and prints the
# CALENDAR_ANCHORS_HARD dict that run_decipherment.py already uses.
# Manual review: confirm before wiring into a full production run.

if alignment:
    print("High-confidence anchors from alignment:")
    for name, entry in alignment.get("anchors", {}).items():
        if not entry.get("ambiguous") and entry.get("confidence", 0) >= 0.87:
            codes = entry.get("anchor_codes_found", [])
            print(f"  {name}: sign(s) {codes}, pos {entry['span']['start_pos']}-"
                  f"{entry['span']['end_pos']}, conf={entry['confidence']:.3f}")
else:
    print("Alignment output not found — run Cell A first.")


In [ ]:
# ── Cell C: Run decipherment with new anchors ────────────────────────────────
# Uses CALENDAR_ANCHORS_HARD already wired in run_decipherment.py.
# Smoke-test mode: 1 chain × 500 iterations for fast wiring check.

result = subprocess.run(
    ["python", "scripts/run_decipherment.py", "--smoke-test"],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Decipherment failed — see stderr above")


In [ ]:
# ── Cell D: Gloss hypotheses ─────────────────────────────────────────────────

result = subprocess.run(
    ["python", "scripts/gloss_hypotheses.py", "--top", "3"],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

GLOSS_HTML = Path(PROJECT_ROOT) / "outputs/analysis/gloss_report.html"
print(f"Gloss report: {GLOSS_HTML}")
try:
    from IPython.display import IFrame
    display(IFrame(str(GLOSS_HTML), width="100%", height=600))
except Exception:
    pass


In [ ]:
# ── Cell E: Reading-order tests v2 ───────────────────────────────────────────

result = subprocess.run(
    ["python", "scripts/reading_order_v2.py", "--tests", "1", "3", "4", "5", "6", "7"],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

RO_HTML = Path(PROJECT_ROOT) / "outputs/analysis/reading_order_v2_report.html"
try:
    from IPython.display import IFrame
    display(IFrame(str(RO_HTML), width="100%", height=700))
except Exception:
    pass


In [ ]:
# ── Cell F: Inpaint damaged glyphs (Zone A required) ────────────────────────
# Skipped gracefully if embeddings_cache.pt does not yet exist.
# Run scripts/train_autoencoder.py first to generate embeddings.

EMBEDDINGS_PATH = Path(PROJECT_ROOT) / "outputs/embeddings_cache.pt"
if EMBEDDINGS_PATH.exists():
    result = subprocess.run(
        ["python", "scripts/inpaint_damaged_glyphs.py", "--max-glyphs", "100"],
        capture_output=True, text=True, cwd=PROJECT_ROOT
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
else:
    print("Embeddings cache not found — skipping inpainting.")
    print("Run scripts/train_autoencoder.py first, then re-run this cell.")


In [ ]:
# ── Cell G: Generate consolidated final report ───────────────────────────────
# Calls generate_final_report.py which reads all pipeline outputs and
# produces a single self-contained HTML with the chart PNG embedded.

result = subprocess.run(
    ["python", "scripts/generate_final_report.py"],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

FINAL_PATH = Path(PROJECT_ROOT) / "outputs/final_report.html"
print(f"Final report: {FINAL_PATH}  ({FINAL_PATH.stat().st_size // 1024} KB)")
try:
    from IPython.display import IFrame
    display(IFrame(str(FINAL_PATH), width="100%", height=900))
except Exception:
    pass


## Quantum Pipeline — IBM Hardware

Cells Q1–Q4: IBMQ credentials → BV IC analysis → Simon decipherment → network centrality → save to Drive.

**Prerequisites:** Setup Cells 1–5 complete, Zone B Cell 2 complete (language models + parallel variants).
Add `IBMQ_TOKEN` and `IBMQ_INSTANCE` as Colab secrets (key icon in the left sidebar).

In [ ]:
# ── Quantum Setup: Load IBMQ credentials and install Qiskit ─────────────────
# Add IBMQ_TOKEN and IBMQ_INSTANCE as Colab secrets before running
# (Runtime sidebar → key icon → 'Add new secret').

import os, subprocess, sys
from google.colab import userdata

os.environ['IBMQ_TOKEN']    = userdata.get('IBMQ_TOKEN')
os.environ['IBMQ_INSTANCE'] = userdata.get('IBMQ_INSTANCE')
os.environ['IBMQ_CHANNEL']  = 'ibm_quantum_platform'

token_len = len(os.environ.get('IBMQ_TOKEN', ''))
print(f"IBMQ_TOKEN    : {'set (' + str(token_len) + ' chars)' if token_len else 'NOT SET'}")
print(f"IBMQ_INSTANCE : {os.environ.get('IBMQ_INSTANCE', 'NOT SET')}")
print(f"IBMQ_CHANNEL  : {os.environ['IBMQ_CHANNEL']}")

def _pip(*args):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *args],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(f'FAILED: {" ".join(args)}\n{r.stderr[-500:]}')
        raise RuntimeError()
    print(f'  ✓ {" ".join(args)}')

_pip('qiskit>=1.0', 'qiskit-ibm-runtime>=0.20')
print('\nQuantum environment ready.')

In [ ]:
# ── Quantum Cell 1: Bernstein-Vazirani IC analysis — IBM hardware ───────────
# Tests whether the rongorongo IC distribution hides a linear Boolean structure
# f(x) = s·x (mod 2).  One quantum oracle query vs 128 classical.
# Output: outputs/quantum/bv_ic_result.json

import os, subprocess, sys
REPO_DIR = '/content/repo'

result = subprocess.run(
    [sys.executable, 'scripts/run_bv_ic_analysis.py', '--backend', 'ibmq'],
    capture_output=True, text=True, cwd=REPO_DIR,
    env={**os.environ},
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('── STDERR ──')
    print(result.stderr[-1000:])

In [ ]:
# ── Quantum Cell 2: Simon's algorithm decipherment — IBM hardware ──────────
# Recovers the hidden XOR period in pre→post-contact sign substitutions
# across the two holy-grail parallel passages.
# Output: outputs/quantum/simon_all_results.json

import os, subprocess, sys
REPO_DIR = '/content/repo'

for passage in ['P007_ADHS', 'P012_ABCDEGHINPQSX']:
    print(f"\n{'='*60}\n  PASSAGE: {passage}\n{'='*60}")
    result = subprocess.run(
        [sys.executable, 'scripts/run_simon_decipherment.py',
         '--backend', 'ibmq', '--passage', passage],
        capture_output=True, text=True, cwd=REPO_DIR,
        env={**os.environ},
    )
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print('── STDERR ──')
        print(result.stderr[-500:])

In [ ]:
# ── Quantum Cell 3: Bigram network centrality analysis ──────────────────────
# Classical PMI-weighted centrality suite across four corpus strata.
# No IBM hardware required — uses Qiskit statevector simulator for
# optional quantum PageRank / Fiedler estimation.
# Output: outputs/network/

import subprocess, sys
REPO_DIR = '/content/repo'

result = subprocess.run(
    [sys.executable, 'scripts/run_network_centrality.py'],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('── STDERR ──')
    print(result.stderr[-1000:])

In [ ]:
# ── Quantum Cell 4: Save quantum results to Drive ───────────────────────────
# Copies outputs/quantum/ and outputs/network/ to Drive.
# Also copies RESULTS.md if generate_final_report.py has already run (Cell G).

import os, shutil
from pathlib import Path

REPO_DIR  = '/content/repo'
DRIVE_OUT = Path('/content/drive/MyDrive/hackingrongo_results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def _save_dir(src: Path, label: str):
    if src.exists():
        dst = DRIVE_OUT / src.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = len(list(dst.rglob('*')))
        print(f'  ✓ {label:45s} ({n} files)')
    else:
        print(f'  — {label:45s} not generated yet')

_save_dir(Path(f'{REPO_DIR}/outputs/quantum'), 'outputs/quantum/')
_save_dir(Path(f'{REPO_DIR}/outputs/network'), 'outputs/network/')

results_md = Path(f'{REPO_DIR}/RESULTS.md')
if results_md.exists():
    shutil.copy(results_md, DRIVE_OUT / 'RESULTS.md')
    print(f'  ✓ RESULTS.md')
else:
    print('  — RESULTS.md not found (run Cell G first)')

print(f'\nQuantum results saved to: {DRIVE_OUT}')